# Moteur de recommandation

> **Description** : ce notebook construit, étape par étape, le moteur de recommandation de l'application. À partir de la sélection de films de l'équipe, complétée par les données TMDB (mots-clés, titre original) et IMDb (réalisateurs, acteurs), chaque film est transformé en une ligne de nombres (genres, année, durée, note, popularité, mots-clés, langue, réalisateurs, acteurs). Un algorithme des plus proches voisins (KNN) trouve ensuite les films qui ressemblent le plus à un film donné.
>
> Chaque étape est expliquée (**Objectif** avant le code, **Constat** après) et les choix sont justifiés. Le code final du moteur sera ensuite repris dans `reco.py`, utilisé par l'application Streamlit.

## 1. Les données

> **Objectif** : charger la sélection de films de l'équipe (`films_selection.csv`), point de départ du catalogue.

In [1]:
import pandas as pd

In [2]:
films = pd.read_csv("../data/films_selection.csv")
print(films.shape)
films.columns

(6942, 13)


Index(['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear',
       'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title',
       'original_language', 'poster_path', 'sans_titre'],
      dtype='str')

> **Objectif** : vérifier que chaque film n'apparaît qu'une fois (identifiant `tconst`). `duplicated()` compte les doublons ; avec `keep=False`, on affiche toutes les versions des films concernés pour les comparer.

In [3]:
doublons = films.duplicated("tconst")
print(doublons.sum())
films[doublons]

0


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre


In [4]:
doublons_keep = films.duplicated("tconst", keep=False)
print(doublons_keep.sum())
films[doublons_keep]

0


,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre


In [5]:
films[doublons_keep][["tconst", "title", "poster_path"]]

,tconst,title,poster_path


> **Constat** : aucun film n'apparaît deux fois : la sélection actuelle est déjà sans doublon. L'ancienne sélection en contenait 3 (voir plus bas, la jointure avec TMDB) ; on garde le code de dédoublonnage par sécurité.

> **Objectif** : supprimer les doublons avec une règle générique, sans numéro de ligne écrit en dur : si un film apparaît plusieurs fois, on garde une seule version, de préférence une qui a une affiche. Méthode : marquer les versions avec affiche, trier pour les faire passer en premier, puis garder la première ligne de chaque film (`drop_duplicates`). S'il n'y a aucun doublon, ce code ne change rien.

In [6]:
films["a_affiche"] = films["poster_path"].notna()
films["a_affiche"].value_counts()

a_affiche
True     6636
False     306
Name: count, dtype: int64

In [7]:
films = films.sort_values("a_affiche", ascending=False)
films

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre,a_affiche
0,tt0035423,movie,Kate & Leopold,0,2001.0,118.0,"Comedy,Fantasy,Romance",6.4,95237,Kate et Léopold,en,/mUvikzKJJSg9khrVdxK8kg3TMHA.jpg,False,True
4445,tt1512235,movie,Super,0,2010.0,96.0,"Action,Comedy,Crime",6.7,88123,Super,en,/iBXsm2VuTLdocOWUHSKjie8fjlU.jpg,False,True
4455,tt1518812,movie,Meek's Cutoff,0,2010.0,104.0,"Adventure,Drama,Thriller",6.5,16652,La dernière piste,en,/p9KtUAD6ltsTkKWeXuqF5GmX7f3.jpg,False,True
4454,tt1517268,movie,Barbie,0,2023.0,114.0,"Adventure,Comedy,Fantasy",6.8,711671,Barbie,en,/iuFNMS8U5cb6xfzi51Dbkovj7vM.jpg,False,True
4453,tt1517260,movie,The Host,0,2013.0,125.0,"Action,Adventure,Drama",5.8,122751,Les âmes vagabondes,en,/ok2sl6rGITZ0W94DeRU4VkB2ssW.jpg,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,tt29008225,movie,Fackham Hall,0,2025.0,97.0,Comedy,6.4,18199,Fackham Hall,NaN,NaN,False,False
5580,tt29028515,movie,Wasteman,0,2025.0,90.0,"Crime,Drama,Thriller",7.2,11297,Wasteman,en,NaN,False,False
4512,tt15459206,movie,Maldoror,0,2024.0,155.0,"Crime,Drama,Thriller",6.6,2284,Le Dossier Maldoror,fr,NaN,False,False
4506,tt15428134,movie,Dunki,0,2023.0,160.0,"Action,Adventure,Comedy",6.5,86923,Dunki,NaN,NaN,False,False


In [8]:
films = films.drop_duplicates(subset="tconst")

In [9]:
films.shape

(6942, 14)

In [10]:
print(films["tconst"].duplicated().sum())
print(films.index.max())

0
6941


> **Constat** : il reste 6 942 lignes et la plus grande étiquette de l'index vaut 6941 : aucune ligne n'a été supprimée, donc pas de « trous » dans l'index. En revanche, le tri a changé l'ordre des lignes (les films sans affiche sont à la fin) : l'étiquette d'un film ne correspond plus à sa position.

> **Objectif** : comprendre, sur un mini-exemple de 3 films, pourquoi un index dont les étiquettes ne suivent pas les positions est dangereux (c'est le cas après un tri, ou après une suppression de lignes, qui laisse des « trous »). pandas utilise deux numérotations : les *étiquettes* de l'index (utilisées par `loc`, `index`, `idxmax`) et les *positions* 0, 1, 2… (utilisées par `iloc`, les tableaux numpy et les résultats d'un modèle). Ici, `mini` a les étiquettes 0, 1, 3 (comme un tableau dont on aurait supprimé une ligne), et `autre` les étiquettes 0, 1, 2.

In [11]:
mini = pd.DataFrame({"film": ["Alien", "Toy Story", "Matrix"]}, index=[0, 1, 3])
autre = pd.DataFrame({"note": [8.4, 8.3, 8.7]})   # index automatique 0, 1, 2
print(mini)
print(autre)

        film
0      Alien
1  Toy Story
3     Matrix
   note
0   8.4
1   8.3
2   8.7


In [12]:
pd.concat([mini, autre], axis=1)

,film,note
0,Alien,8.4
1,Toy Story,8.3
3,Matrix,NaN
2,NaN,8.7


> **Constat** : `pd.concat(axis=1)` associe les lignes par étiquette, pas par position. Avec des étiquettes différentes (0, 1, 3 d'un côté ; 0, 1, 2 de l'autre), on obtient 4 lignes au lieu de 3 et des `NaN` : Matrix perd sa note, qui part sur une ligne fantôme.

In [13]:
i = mini.index[mini["film"] == "Matrix"][0]
print(i)    # va donner 3
print(autre)
# autre.iloc[[i]]   # provoque volontairement une IndexError (étiquette prise pour une position)

3
   note
0   8.4
1   8.3
2   8.7


> **Constat** : `mini.index[…]` renvoie une *étiquette* (3), alors que `iloc` attend une *position* (0, 1 ou 2) : erreur `IndexError`. Si la position avait existé, on aurait obtenu un autre film, sans message d'erreur.
>
> **Conclusion** : si les étiquettes ne suivent pas les positions, deux bugs menacent la suite : un `pd.concat` décalerait les lignes et créerait des `NaN` ; un film repéré par son étiquette puis cherché par sa position renverrait le mauvais film. Il faut donc que étiquette = position : on renumérote de 0 à n − 1.

> **Objectif** : réparer l'index. On retire la colonne de travail `a_affiche`, on remet les films dans leur ordre d'origine (`sort_index`), puis on renumérote de 0 à n − 1 (`reset_index(drop=True)`) pour que étiquette = position. La jointure plus loin renumérote aussi les lignes, mais on garde cette étape par sécurité : le code reste juste même si l'on change l'ordre des opérations.

In [14]:
films = films.drop(columns="a_affiche")

In [15]:
films = films.sort_index()

In [16]:
films = films.reset_index(drop=True)  # drop=True jette l’ancien index au lieu d’en faire une nouvelle colonne

In [17]:
print(films.shape)
print(films.index.min(), films.index.max())

(6942, 13)
0 6941


> **Constat** : 6 942 films, avec un index de 0 à 6 941 : étiquette = position.

In [18]:
liste_des_colonnes = list(films.columns)
print(liste_des_colonnes)
print(len(liste_des_colonnes))

['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title', 'original_language', 'poster_path', 'sans_titre']
13


> **Constat** : classement des 13 colonnes de la sélection.
> - *Décrivent le film* (utiles pour mesurer une ressemblance) : `genres`, `startYear`, `runtimeMinutes`, `averageRating`, `numVotes`, `original_language`.
> - *Affichage* : `title` (titre FR), `primaryTitle` (titre international), `poster_path` (affiche) ; `tconst` est l'identifiant (clé de jointure).
> - *Autres* : `titleType` (toujours « movie ») et `isAdult` (toujours 0) sont constantes après filtrage ; `sans_titre` indique les films sans titre FR.
>
> Il manque une description du *contenu* du film (de quoi il parle) : ce sera le rôle des mots-clés de TMDB.

### Le fichier TMDB

In [19]:
apercu = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", nrows=5)
print(apercu.shape)
apercu.columns

(5, 24)


Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords'],
      dtype='str')

> **Objectif** : ne charger que les colonnes utiles de TMDB (5 sur 24) :
> - `keywords` : les mots-clés du film ;
> - `original_title` : le titre en version originale ;
> - `imdb_id` : l'identifiant IMDb (`tt…`), le même que `tconst` dans notre sélection ; c'est la clé de jointure ;
> - `poster_path` et `popularity` : serviront à départager les doublons de TMDB (affiche d'abord, popularité ensuite).
>
> On ne charge que ces 5 colonnes : moins de mémoire et lecture plus rapide.

In [20]:
colonnes_tmdb = ["imdb_id", "keywords", "original_title", "poster_path", "popularity"]

In [21]:
tmdb = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=colonnes_tmdb)
tmdb.shape

(1529949, 5)

> **Objectif** : retirer les films TMDB sans identifiant IMDb (`imdb_id` vide). Sans identifiant, ils ne pourraient jamais être joints à notre sélection, dont tous les films viennent d'IMDb : on ne perd donc rien.

In [22]:
tmdb["imdb_id"].isna().sum()

np.int64(836561)

In [23]:
tmdb = tmdb.dropna(subset=["imdb_id"])
tmdb.shape

(693388, 5)

> **Constat** : 836 561 films sur 1 529 949 n'ont pas d'identifiant IMDb (plus de la moitié, surtout des films très confidentiels, absents d'IMDb). Il reste 693 388 films.

> **Objectif** : repérer les films qui apparaissent plusieurs fois dans TMDB (même `imdb_id`), puis regarder de près ceux qui concernent nos films.

In [24]:
# doublons dans tmdb ?
tmdb["imdb_id"].duplicated().sum()

np.int64(5633)

In [25]:
doublons_tmdb = tmdb[tmdb["imdb_id"].duplicated(keep=False)]

In [26]:
doublons_nos_films = doublons_tmdb[doublons_tmdb["imdb_id"].isin(films["tconst"])]
doublons_nos_films["imdb_id"].nunique()

3

In [27]:
doublons_nos_films.sort_values("imdb_id")

,imdb_id,original_title,popularity,poster_path,keywords
508610,tt1756855,Coyote vs. Acme,1.400,NaN,"lawsuit, live action and animation, based on m..."
1429111,tt1756855,Coyote v. Acme,2.069,NaN,NaN
529903,tt23468450,Longlegs,0.000,NaN,"fbi, serial killer, female protagonist, occult"
1295936,tt23468450,Longlegs,1.400,/3wVnnLzX5eObFWzrsy5vJGpMCCI.jpg,NaN
612321,tt27441481,கூலி,0.000,/3GUVDQ0arZcZzxRAVfyAxijTKEe.jpg,NaN
1325326,tt27441481,Thalaivar 171,0.618,/hDntDhxW0uKsCQRaUfcwCyYUXKY.jpg,NaN


> **Constat** : 5 633 identifiants `imdb_id` apparaissent plusieurs fois, dont 3 concernent nos films (Coyote vs. Acme, Longlegs et Coolie, alias « Thalaivar 171 »). Ce sont les 3 films qui étaient en double dans l'ancienne sélection : elle avait hérité de ces doublons TMDB lors de sa propre jointure. La sélection actuelle est corrigée, mais notre jointure avec TMDB doit, elle aussi, les éviter. Souvent, une version est une fiche complète et l'autre une fiche vide ou provisoire (popularité proche de 0, titre de travail comme « Thalaivar 171 »).

> **Objectif** : garder une seule version par `imdb_id` : d'abord celle qui a une affiche, puis, à égalité, la plus populaire (tri sur deux colonnes, puis `drop_duplicates`). Si aucune version n'a d'affiche, toutes valent `False` pour le premier critère : c'est alors la popularité qui décide.

In [28]:
tmdb["a_affiche"] = tmdb["poster_path"].notna()

In [29]:
tmdb = tmdb.sort_values(["a_affiche", "popularity"], ascending=False)

In [30]:
tmdb = tmdb.drop_duplicates(subset="imdb_id")

In [31]:
tmdb = tmdb.drop(columns="a_affiche")
print(tmdb.shape, tmdb["imdb_id"].duplicated().sum())

(687755, 5) 0


> **Constat** : il reste 687 755 films, sans doublon. *Coyote vs. Acme*, qui n'a d'affiche dans aucune version, reste sans affiche.

### La jointure

> **Objectif** : ajouter à chacun de nos films ses mots-clés (`keywords`) et son titre VO (`original_title`), en associant `tconst` (sélection) et `imdb_id` (TMDB). On n'emporte pas `poster_path`, déjà présent dans la sélection (sinon pandas créerait `poster_path_x` et `poster_path_y`). Jointure à gauche : on garde tous nos films, même ceux absents de TMDB.

In [32]:
tmdb_a_joindre = tmdb[["imdb_id", "keywords", "original_title"]]

In [33]:
films = films.merge(tmdb_a_joindre, left_on="tconst", right_on="imdb_id", how="left")
films.shape

(6942, 16)

In [34]:
# Combien de films sont absents de TMDB ? Après un left join, un imdb_id vide veut dire « pas trouvé ».
films["imdb_id"].isna().sum()

np.int64(184)

In [35]:
# Années de sortie des films absents de TMDB
films[films["imdb_id"].isna()]["startYear"].value_counts()

startYear
2025.0    88
2026.0    64
2024.0    30
2023.0     2
Name: count, dtype: int64

In [36]:
films = films.drop(columns="imdb_id")
films.shape

(6942, 15)

> **Constat** : toujours 6 942 lignes (aucun film perdu ni dupliqué, grâce au dédoublonnage de TMDB). 184 films sont absents de TMDB, dont 182 sortis en 2024-2026, après l'extraction du fichier TMDB (instantané figé) ; ils n'ont ni mots-clés, ni titre VO, ni affiche.

> **Objectif** : compter les films sans mots-clés et voir de quelles années ils datent.

In [37]:
# Films sans mots-clés
films["keywords"].isna().sum()

np.int64(652)

In [38]:
filtre_sans_mots_cles = films["keywords"].isna()
films[filtre_sans_mots_cles].head()

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,title,original_language,poster_path,sans_titre,keywords,original_title
39,tt0066798,movie,Money Money Money,0,1972.0,113.0,"Adventure,Comedy,Crime",6.9,2900,"L'aventure, c'est l'aventure",fr,/f04ZjhayRhsCHzHNrc7wvglUWFW.jpg,False,NaN,L'Aventure c'est l'aventure
84,tt0068519,movie,Gentlemen of Fortune,0,1971.0,88.0,"Comedy,Crime,Drama",8.3,14706,Les Gentilshommes de la chance,ru,/owZ2KQ2x99XqMprSvFNeTdmCBk8.jpg,False,NaN,Джентльмены удачи
134,tt0070356,movie,Now Where Did the Seventh Company Get to?,0,1973.0,95.0,"Action,Comedy,War",6.9,2958,Mais où est donc passée la 7ème compagnie ?,fr,/clSsWDMsUJ49ptNkhJ0yLWyPXlU.jpg,False,NaN,Mais où est donc passée la 7ème compagnie ?
136,tt0070453,movie,Wedding in Blood,0,1973.0,95.0,"Crime,Drama,Thriller",6.9,2540,Les noces rouges,fr,/1BAC8HGlfSfZobuN3WGzjaieZDt.jpg,False,NaN,Les Noces rouges
148,tt0070827,movie,Shock Treatment,0,1973.0,91.0,"Crime,Drama,Horror",6.1,2202,Traitement de choc,fr,/4yGtwKEGOzgOpaZnAi3aQ0nII4p.jpg,False,NaN,Traitement de choc


In [39]:
films[filtre_sans_mots_cles]["startYear"].value_counts().head(10)

startYear
2025.0    128
2026.0     91
2024.0     81
2023.0     46
2022.0     37
2019.0     22
2018.0     18
2021.0     16
2017.0     16
2020.0     15
Name: count, dtype: int64

> **Constat** : 652 films, surtout récents (2025 : 128 ; 2026 : 91 ; 2024 : 81 ; 2023 : 46 ; 2022 : 37). Ce sont les 184 films absents de TMDB, plus des films présents dans TMDB mais que la communauté n'a pas encore décrits par des mots-clés.

> **Objectif** : donner aux films sans mots-clés leurs genres comme mots-clés, en minuscules et au même format que TMDB (« action, crime »).
>
> Pourquoi ne pas les laisser vides ? Intuition : un film sans mots-clés ne pourrait être comparé aux autres que par ses genres et ses chiffres ; il risque de fausser les ressemblances (à vérifier lors du calcul des distances).

In [40]:
genres_texte = films["genres"].str.lower().str.replace(",", ", ")

In [41]:
print(genres_texte.shape)
genres_texte.head()

(6942,)


0     comedy, fantasy, romance
1     drama, mystery, thriller
2             sci-fi, thriller
3    adventure, drama, romance
4       drama, horror, mystery
Name: genres, dtype: str

In [42]:
films["keywords"] = films["keywords"].fillna(genres_texte)

In [43]:
films["keywords"].isna().sum()

np.int64(0)

> **Constat** : il ne reste aucun film sans mots-clés.

> **Objectif** : enregistrer le résultat dans `data/films_prepares.csv`. La préparation (lire les 674 Mo de TMDB, nettoyer, joindre) n'est ainsi faite qu'une fois : l'appli lira directement ce petit fichier, au lieu de tout refaire à chaque démarrage.

In [44]:
films.to_csv("../data/films_prepares.csv", index=False)

In [45]:
# Vérification
pd.read_csv("../data/films_prepares.csv").shape

(6942, 15)

> **Constat** : le fichier se relit correctement : 6 942 films, 15 colonnes.

## 2. Un film en nombres

> **Objectif** : un calcul de distance ne fonctionne qu'avec des nombres. On transforme donc chaque film en une ligne de nombres : d'abord ses genres (en colonnes 0/1), puis ses caractéristiques numériques (année, durée, note, nombre de votes).

> **Remarque** : on s'attend à trouver plus de 7 genres. Le filtre garde un film dès qu'il possède *au moins un* des 7 genres visés (Action, Adventure, Comedy, Thriller, Horror, Sci-Fi, War), mais ses autres genres (Drama, Crime, Romance…) restent dans la colonne `genres`.

> **Objectif** : lister tous les genres présents, en découpant chaque texte `genres` sur les virgules.

In [46]:
liste_genres = []
for texte in films["genres"]:
    for genre in texte.split(","):
        if genre not in liste_genres:
            liste_genres.append(genre)

In [47]:
print("Nb de genres :", len(liste_genres))

Nb de genres : 21


In [48]:
sorted(liste_genres)

['Action',
 'Adventure',
 'Animation',
 'Biography',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Family',
 'Fantasy',
 'History',
 'Horror',
 'Music',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Sport',
 'Thriller',
 'War',
 'Western']

> **Objectif** : transformer la colonne texte `genres` en une colonne par genre, avec 1 si le film a ce genre et 0 sinon. `str.get_dummies(sep=",")` fait ce travail en une ligne : il découpe chaque texte sur les virgules (comme notre boucle) et crée automatiquement une colonne par genre rencontré. Cela va donc créer 21 colonnes (avec des 0 et des 1).

In [49]:
genres = films["genres"].str.get_dummies(sep=",")
genres.shape

(6942, 21)

In [50]:
genres.head()

,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Horror,Music,Musical,Mystery,Romance,Sci-Fi,Sport,Thriller,War,Western
0,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,1,0,0,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
3,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,...,1,0,0,1,0,0,0,0,0,0


> **Constat** : 6 942 lignes (une par film) et 21 colonnes (une par genre), comme le nombre trouvé avec la boucle. Chaque ligne contient des 1 pour les genres du film et des 0 partout ailleurs.

In [51]:
nb_genres = genres.sum(axis=1)
nb_genres.value_counts()

3    5057
2    1560
1     325
Name: count, dtype: int64

> **Constat** : la grande majorité des films ont 3 genres (5 057), 1 560 en ont 2 et seulement 325 n'en ont qu'un. Deux films se ressembleront donc par plusieurs genres à la fois, pas seulement par un genre principal.

> **Objectif** : comparer avec `pd.get_dummies`, qui crée aussi des colonnes 0/1, mais sans découper le texte sur les virgules.

In [52]:
pd.get_dummies(films["genres"]).shape

(6942, 356)

> **Constat** : 356 colonnes au lieu de 21. `pd.get_dummies` prend chaque texte entier comme une seule catégorie : « Action,Crime,Drama » et « Action,Drama » deviennent deux colonnes distinctes. Deux films partageant Action et Drama n'auraient alors rien en commun. `str.get_dummies(sep=",")` découpe d'abord le texte, et chaque genre devient une colonne : c'est ce qu'il nous faut, puisqu'un film peut avoir plusieurs genres.

> **Remarque (vocabulaire)** :
> - *one-hot encoding* : chaque film appartient à **une seule** catégorie, donc il y a exactement un 1 par ligne (c'est ce que fait `pd.get_dummies`, par exemple pour la langue originale) ;
> - *multi-hot encoding* : un film peut appartenir à **plusieurs** catégories à la fois, donc une ligne peut contenir plusieurs 1 (c'est ce que fait `str.get_dummies(sep=",")` pour les genres : jusqu'à trois 1 par film).

> **Objectif** : regrouper les 4 caractéristiques chiffrées des films : année, durée, note et nombre de votes.

In [53]:
numeriques = films[["startYear", "runtimeMinutes", "averageRating", "numVotes"]]
numeriques.info()

<class 'pandas.DataFrame'>
RangeIndex: 6942 entries, 0 to 6941
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   startYear       6942 non-null   float64
 1   runtimeMinutes  6942 non-null   float64
 2   averageRating   6942 non-null   float64
 3   numVotes        6942 non-null   int64  
dtypes: float64(3), int64(1)
memory usage: 217.1 KB


> **Objectif** : afficher l'année et la durée en nombres entiers (1971 au lieu de 1971.0). On vérifie d'abord qu'il n'y a aucune valeur manquante, car un entier ne peut pas valoir NaN.

In [54]:
films[["startYear", "runtimeMinutes"]].isna().sum()

startYear         0
runtimeMinutes    0
dtype: int64

> **Constat** : aucune valeur manquante, la conversion en entiers est possible. Cela ne change rien aux calculs : c'est seulement plus lisible.

In [55]:
films["startYear"] = films["startYear"].astype(int)
films["runtimeMinutes"] = films["runtimeMinutes"].astype(int)

In [56]:
numeriques = films[["startYear", "runtimeMinutes", "averageRating", "numVotes"]]
numeriques.info()

<class 'pandas.DataFrame'>
RangeIndex: 6942 entries, 0 to 6941
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   startYear       6942 non-null   int64  
 1   runtimeMinutes  6942 non-null   int64  
 2   averageRating   6942 non-null   float64
 3   numVotes        6942 non-null   int64  
dtypes: float64(1), int64(3)
memory usage: 217.1 KB


> **Objectif** : regarder les ordres de grandeur de ces 4 colonnes (minimum, maximum, moyenne…) pour repérer celles qui poseront problème dans un calcul de distance.

In [57]:
numeriques.describe()

,startYear,runtimeMinutes,averageRating,numVotes
count,6942.000000,6942.000000,6942.000000,6.942000e+03
mean,2006.049121,110.139297,6.786488,1.284018e+05
std,13.943545,21.298099,0.643482,2.315260e+05
min,1970.000000,63.000000,1.800000,2.001000e+03
25%,1997.000000,96.000000,6.300000,1.726250e+04
50%,2009.000000,106.000000,6.700000,4.530000e+04
75%,2017.000000,119.000000,7.200000,1.370928e+05
max,2026.000000,566.000000,9.300000,3.267800e+06


> **Constat** : les 4 colonnes ont des échelles très différentes :
> - année : de 1970 à 2026 (écart de 56) ;
> - durée : de 63 à 566 minutes ; la moitié des films durent entre 96 et 119 minutes, le maximum (566) est donc très isolé ;
> - note : de 1,8 à 9,3, mais la moitié des films sont notés entre 6,3 et 7,2 : les notes très basses sont rares ;
> - votes : de 2 001 à 3 267 800.
>
> Le nombre de votes est le plus problématique. Ses valeurs sont énormes, et très déséquilibrées : la moyenne (128 402) est presque trois fois plus grande que la médiane (45 300), signe que quelques films à plusieurs millions de votes tirent la moyenne vers le haut.

> **Objectif** : remplacer le nombre de votes par son logarithme en base 10 (10 000 → 4, 100 000 → 5, 1 000 000 → 6). L'écart entre les films très populaires et les autres devient beaucoup plus raisonnable.

In [58]:
import numpy as np

numeriques["log_votes"] = np.log10(numeriques["numVotes"])

> **Remarque** : ajouter une colonne à `numeriques` ne modifie pas `films`. Depuis pandas 3, une table extraite d'une autre (`films[[...]]`) se comporte toujours comme une copie indépendante. Avec pandas 2, il fallait souvent ajouter `.copy()` pour éviter un avertissement.

In [59]:
print(films.columns)
print(numeriques.columns)

Index(['tconst', 'titleType', 'primaryTitle', 'isAdult', 'startYear',
       'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'title',
       'original_language', 'poster_path', 'sans_titre', 'keywords',
       'original_title'],
      dtype='str')
Index(['startYear', 'runtimeMinutes', 'averageRating', 'numVotes',
       'log_votes'],
      dtype='str')


> **Constat** : `log_votes` a bien été ajoutée à `numeriques`, et `films` est resté inchangé.

> **Objectif** : vérifier l'effet du logarithme sur l'échelle des votes : le minimum et le maximum doivent être proches de 3,3 (2 001 votes) et de 6,5 (3 267 800 votes).

In [60]:
numeriques[["log_votes", "numVotes"]].describe()

,log_votes,numVotes
count,6942.000000,6.942000e+03
mean,4.698559,1.284018e+05
std,0.601951,2.315260e+05
min,3.301247,2.001000e+03
25%,4.237104,1.726250e+04
50%,4.656098,4.530000e+04
75%,5.137014,1.370928e+05
max,6.514255,3.267800e+06


> **Constat** : avec le logarithme, les votes vont de 3,30 (2 001 votes) à 6,51 (3 267 800 votes). La moyenne (4,70) est maintenant proche de la médiane (4,66) : les quelques films à plusieurs millions de votes n'écrasent plus les autres.

> **Objectif** (curiosité, non indispensable) : trier les films par nombre de votes, pour voir d'un coup les moins votés (début du tableau) et les plus votés (fin du tableau).

In [61]:
films.sort_values("numVotes")[["title", "startYear", "averageRating", "numVotes"]]

,title,startYear,averageRating,numVotes
6545,Petit Paysan,2017,7.0,2001
290,La vie devant soi,1977,7.1,2002
181,Que la fete commence,1975,7.0,2003
2890,Jacquou le croquant,2007,6.4,2006
6679,Chanson douce,2019,6.0,2010
...,...,...,...,...
3308,Interstellar,2014,8.7,2638221
1939,Fight Club,1999,8.8,2686520
4206,Inception,2010,8.8,2892496
3104,The Dark Knight : Le Chevalier noir,2008,9.1,3253541


> **Constat** : les films les moins votés ont à peine plus de 2 000 votes ; ce sont des films français (Petit Paysan, La vie devant soi, Que la fete commence…), retenus grâce au seuil de votes plus bas que l'équipe a fixé pour les films français. Les plus votés sont des films très connus (Interstellar, Fight Club, Inception, The Dark Knight, Les Évadés) et aussi très bien notés (de 8,7 à 9,3). Les Évadés est à la fois le film le plus voté (3 267 800 votes) et le mieux noté de la sélection (9,3, la note maximale vue dans `describe()`).

> **Objectif** : trier les films par durée, pour voir les plus courts et surtout les plus longs, et décider quoi faire des durées exceptionnelles.

In [62]:
films.sort_values("runtimeMinutes")[["title", "startYear", "averageRating", "runtimeMinutes"]]

,title,startYear,averageRating,runtimeMinutes
4340,Winnie l'Ourson,2011,7.1,63
4826,Who Killed Captain Alex?,2010,7.7,64
5535,Yannick,2023,7.0,67
951,Tetsuo,1989,6.9,67
2679,Interstella 5555,2003,7.8,68
...,...,...,...,...
1307,Gettysburg: la dernière bataille,1993,7.6,254
1347,Smoking/No Smoking,1993,7.3,298
4943,Gangs of Wasseypur,2012,8.2,321
1023,La Révolution française,1989,7.6,335


In [63]:
films.sort_values("runtimeMinutes")[["title", "startYear", "averageRating", "runtimeMinutes"]].tail(15)

,title,startYear,averageRating,runtimeMinutes
4205,Animal,2023,6.2,204
259,Le Message,1976,8.9,207
5519,Ramayana: L'avènement d'une légende,2026,8.2,208
3018,Jodhaa Akbar,2008,7.5,213
5793,Dhurandhar,2025,8.2,214
420,La Porte du paradis,1980,6.7,219
6010,Dhurandhar: The Revenge,2026,8.2,235
3753,Love Exposure,2008,8.0,237
3979,La Ligue des Justiciers de Zack Snyder,2021,7.9,242
6415,Kill Bill: The Whole Bloody Affair,2004,8.7,253


> **Constat** : les durées les plus longues sont réelles (Shoah : 566 minutes, Gangs of Wasseypur : 321 minutes…), ce ne sont pas des erreurs. Mais ces films sont exceptionnels : d'après le `describe()`, le premier quartile (ligne `25%`) est à 96 minutes et le troisième quartile (ligne `75%`) à 119 minutes, donc la moitié des films durent entre 96 et 119 minutes.

> **Décision** : on garde ces films, mais on plafonnera leur durée à 180 minutes (3 heures). Au-delà, « très long » suffit : Shoah n'a pas besoin d'être considéré comme trois fois plus long qu'un film de 3 heures. Sinon, ce seul film étirerait toute l'échelle des durées.

## 3. Une même échelle pour tous

> **Objectif** : comprendre, sur deux films qui se ressemblent (Alien et sa suite Aliens), pourquoi des colonnes d'échelles très différentes faussent le calcul de distance.

> **Objectif** : retrouver Alien et sa suite parmi les films dont le titre contient « Alien », pour récupérer leurs identifiants (`tconst`).

In [64]:
filtre_alien = films["title"].str.contains("Alien")
films[filtre_alien][["tconst", "title", "original_title", "startYear"]]

,tconst,title,original_title,startYear
352,tt0078748,"Alien, le 8ème passager",Alien,1979
741,tt0090605,"Aliens, le retour",Aliens,1986
1206,tt0103644,Alien 3,Alien³,1992
1675,tt0118583,"Alien, la résurrection",Alien Resurrection,1997
2687,tt0370263,Alien vs. Predator,AVP: Alien vs. Predator,2004
3220,tt0758730,Aliens vs Predator: Requiem,Aliens vs Predator: Requiem,2007
3397,tt0892782,Monstres contre Aliens,Monsters vs Aliens,2009
4860,tt18412256,Alien: Romulus,Untitled Alien Film,2024
5234,tt2316204,Alien: Covenant,Alien: Covenant,2017


> **Constat** : plusieurs films de la saga contiennent « Alien » dans leur titre. Les deux qui nous intéressent sont Alien, le 8ème passager (1979, `tt0078748`) et sa suite Aliens, le retour (1986, `tt0090605`).

In [65]:
deux_films = films[films["tconst"].isin(["tt0078748", "tt0090605"])]
deux_films[["title", "startYear", "runtimeMinutes", "averageRating", "numVotes"]]

,title,startYear,runtimeMinutes,averageRating,numVotes
352,"Alien, le 8ème passager",1979,117,8.4,1084329
741,"Aliens, le retour",1986,137,8.4,853283


In [66]:
numeriques[films["tconst"].isin(["tt0078748", "tt0090605"])]

,startYear,runtimeMinutes,averageRating,numVotes,log_votes
352,1979,117,8.4,1084329,6.035161
741,1986,137,8.4,853283,5.931093


> **Constat** : les deux films ont la même note (8,4) et des popularités proches (`log_votes` : 6,04 et 5,93). Ils diffèrent surtout par la durée (20 minutes d'écart) et par l'année (7 ans d'écart).

> **Objectif** : calculer la distance euclidienne entre Alien et Aliens sur les 4 colonnes retenues (année, durée, note, `log_votes`), en regardant la part de chaque colonne dans le résultat.

In [67]:
colonnes = ["startYear", "runtimeMinutes", "averageRating", "log_votes"]
les_deux_aliens = numeriques[films["tconst"].isin(["tt0078748", "tt0090605"])][colonnes]
ecarts = les_deux_aliens.iloc[0] - les_deux_aliens.iloc[1]
ecarts

startYear         -7.000000
runtimeMinutes   -20.000000
averageRating      0.000000
log_votes          0.104068
dtype: float64

In [68]:
ecarts ** 2

startYear          49.00000
runtimeMinutes    400.00000
averageRating       0.00000
log_votes           0.01083
dtype: float64

In [69]:
print(f"Distance euclidienne entre les deux films : {np.sqrt((ecarts ** 2).sum()):.2f}")

Distance euclidienne entre les deux films : 21.19


> **Constat** : la distance vaut 21,19. Sur les 449 de la somme des carrés, la durée en apporte 400 (environ 89 %) et l'année 49 ; la note et la popularité ne comptent quasiment pas. Avec les valeurs brutes, la distance mesure donc surtout un écart de durée. Un écart de 20 minutes pèse beaucoup plus qu'un écart de note de 1 point, alors que pour un spectateur, la note compte au moins autant.

> **Remarque** : prenons deux films identiques en tout, sauf l'année (10 ans d'écart) : la distance vaut 10. Prenons maintenant deux films identiques en tout, sauf la note (1 point d'écart) : la distance vaut 1. L'année pèse donc 10 fois plus. Pourtant, 1 point de note couvre 13 % de l'échelle des notes (de 1,8 à 9,3), et 10 ans 18 % de la période (1970–2026) : deux écarts du même ordre. La distance brute dépend donc des unités de chaque colonne, pas de ce qui compte pour un spectateur : c'est un problème pour la recommandation.

> **Objectif** : ramener les 4 colonnes chiffrées entre 0 et 1 avec la normalisation min-max : chaque valeur x devient (x − min) / (max − min), avec le min et le max de sa colonne. Le plus petit devient 0, le plus grand 1, le reste se range entre les deux. On procède en deux étapes :
> 1. à la main, sur nos deux films seulement, pour comprendre la formule et voir son effet sur la distance ;
> 2. automatiquement, sur tout le catalogue, avec `MinMaxScaler` de scikit-learn.

> **Objectif de l'étape 1** : calculer le min et le max de chaque colonne sur tout le catalogue (avec le plafond de 180 minutes pour la durée), normaliser Alien et Aliens, puis recalculer leur distance.

In [70]:
mini = numeriques[colonnes].min()
maxi = numeriques[colonnes].max()
maxi["runtimeMinutes"] = 180   # plafond décidé
pd.DataFrame({"min": mini, "max": maxi})

,min,max
startYear,1970.000000,2026.000000
runtimeMinutes,63.000000,180.000000
averageRating,1.800000,9.300000
log_votes,3.301247,6.514255


In [71]:
les_deux_norm = (les_deux_aliens - mini) / (maxi - mini)
les_deux_norm

,startYear,runtimeMinutes,averageRating,log_votes
352,0.160714,0.461538,0.88,0.850889
741,0.285714,0.632479,0.88,0.818500


In [72]:
ecarts_norm = les_deux_norm.iloc[0] - les_deux_norm.iloc[1]
print(f"Distance après normalisation : {np.sqrt((ecarts_norm ** 2).sum()):.2f}")

Distance après normalisation : 0.21


> **Constat** : la distance vaut maintenant 0,21. On ne peut pas la comparer à 21,19 : les unités ont changé. Pour savoir si la normalisation a vraiment corrigé le problème, il faut regarder colonne par colonne.

> **Objectif** : mesurer, avant et après normalisation, la part de chaque colonne dans la distance, ainsi que la taille de chaque écart par rapport à l'étendue de sa colonne.

In [73]:
ecarts_norm.abs()

startYear         0.12500
runtimeMinutes    0.17094
averageRating     0.00000
log_votes         0.03239
dtype: float64

In [74]:
(ecarts_norm ** 2) / (ecarts_norm ** 2).sum()

startYear         0.340454
runtimeMinutes    0.636688
averageRating     0.000000
log_votes         0.022859
dtype: float64

In [75]:
(ecarts ** 2) / (ecarts ** 2).sum()

startYear         0.109129
runtimeMinutes    0.890847
averageRating     0.000000
log_votes         0.000024
dtype: float64

> **Constat** : la normalisation a réglé le problème des unités : chaque écart est mesuré par rapport à l'étendue de sa colonne (20 minutes représentent 17 % de l'étendue des durées, 7 ans 12,5 % de la période). Mais la durée apporte encore environ 64 % de la somme des carrés (contre 89 % avant normalisation), l'année 34 %, la popularité 2 % et la note 0 %.
>
> Ce n'est pas un défaut de calcul : ces deux films diffèrent réellement surtout par leur durée, et le carré amplifie le plus grand écart. Reste une vraie question, que la normalisation ne tranche pas : *faut-il que 20 minutes de durée comptent autant pour juger de la ressemblance entre deux films ?* Deux pistes pour la suite : l'ajout des genres, des mots-clés et de la langue diluera le poids de ces 4 colonnes ; et on pourra donner un poids plus faible aux colonnes chiffrées si elles comptent trop.

> **Objectif de l'étape 2** : appliquer la normalisation min-max à tout le catalogue avec `MinMaxScaler` de scikit-learn. D'abord sans plafond sur la durée, pour mesurer l'effet des films très longs.

In [76]:
from sklearn.preprocessing import MinMaxScaler

In [77]:
sans_plafond = MinMaxScaler().fit_transform(numeriques[colonnes])
sans_plafond = pd.DataFrame(sans_plafond, columns=colonnes)
sans_plafond.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6942.000000,6942.000000,6942.000000,6942.000000
mean,0.643734,0.093716,0.664865,0.434892
std,0.248992,0.042342,0.085798,0.187348
min,0.000000,0.000000,0.000000,0.000000
25%,0.482143,0.065606,0.600000,0.291271
50%,0.696429,0.085487,0.653333,0.421677
75%,0.839286,0.111332,0.720000,0.571355
max,1.000000,1.000000,1.000000,1.000000


> **Constat** : sans plafond, la durée normalisée a une moyenne de 0,094, et les trois quarts des films sont en dessous de 0,111 (ligne `75%`). Shoah (566 minutes) occupe la valeur 1 et étire toute l'échelle : les films « normaux » sont écrasés près de 0, et un écart d'une demi-heure entre deux d'entre eux ne compte presque plus. L'année et la popularité sont bien mieux réparties entre 0 et 1.
>
> **Remarque** : la note montre le même phénomène, en sens inverse. Quelques films très mal notés (1,8 au minimum) étirent l'échelle vers le bas : la moitié des films ont une note normalisée entre 0,60 et 0,72. Un écart d'un point entre deux films ne vaut donc que 0,13 environ. On n'y touche pas ici, mais un plancher sur la note (comme le plafond sur la durée) serait une piste.

> **Objectif** : appliquer la décision prise plus haut (durée plafonnée à 180 minutes), puis refaire la normalisation et comparer la durée normalisée. La méthode `clip(upper=180)` remplace toute valeur supérieure à 180 par 180, et laisse les autres inchangées.

In [78]:
numeriques["runtimeMinutes"] = numeriques["runtimeMinutes"].clip(upper=180)
numeriques["runtimeMinutes"].max()

np.int64(180)

In [79]:
numeriques_norm = MinMaxScaler().fit_transform(numeriques[colonnes])
numeriques_norm = pd.DataFrame(numeriques_norm, columns=colonnes)
numeriques_norm.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6942.000000,6942.000000,6942.000000,6942.000000
mean,0.643734,0.400888,0.664865,0.434892
std,0.248992,0.168006,0.085798,0.187348
min,0.000000,0.000000,0.000000,0.000000
25%,0.482143,0.282051,0.600000,0.291271
50%,0.696429,0.367521,0.653333,0.421677
75%,0.839286,0.478632,0.720000,0.571355
max,1.000000,1.000000,1.000000,1.000000


> **Constat** : avec le plafond, la moyenne de la durée normalisée passe de 0,094 à 0,40. Les films « normaux » occupent maintenant une bonne partie de l'échelle de 0 à 1, au lieu d'être écrasés près de 0 : un écart d'une demi-heure entre deux films redevient visible. Les films de plus de 3 heures valent tous 1 (« très long »).

> **Remarque** : une autre solution aurait été de prendre le logarithme de la durée, comme pour les votes. On a préféré le plafond : il est plus simple à expliquer (« au-delà de 3 heures, c'est très long ») et il garde les durées ordinaires exactement proportionnelles.

> **Remarque (pour la suite)** : les genres sont codés en 0 ou 1, donc un écart sur un genre vaut au plus 1. Avec la normalisation, c'est aussi le cas pour chaque colonne chiffrée. Quand on réunira genres et colonnes chiffrées dans une même distance, aucune colonne n'écrasera les autres à cause de son unité. Sans normalisation, 20 minutes d'écart compteraient autant que 20 genres différents.

> **Objectif** : vérifier que deux colonnes chiffrées ne disent pas la même chose. Si deux colonnes sont très corrélées, l'information qu'elles portent compte deux fois dans la distance. La corrélation va de -1 à 1 : plus elle est proche de 1 (ou de -1), plus les deux colonnes varient ensemble.

In [80]:
numeriques_norm.corr().round(2)

,startYear,runtimeMinutes,averageRating,log_votes
startYear,1.00,0.13,-0.13,0.07
runtimeMinutes,0.13,1.00,0.30,0.22
averageRating,-0.13,0.30,1.00,0.22
log_votes,0.07,0.22,0.22,1.00


> **Constat** : la plus forte corrélation est de 0,30 (entre la durée et la note) ; toutes les autres sont inférieures ou égales à 0,22 en valeur absolue. Aucune colonne ne double une autre : on garde les quatre.

## 4. Les films voisins

> **Objectif** : réunir, pour chaque film, ses 21 colonnes de genres et ses 4 colonnes chiffrées normalisées en une seule ligne de 25 nombres. C'est sur ces lignes qu'on calculera les distances entre films.

In [81]:
X1 = pd.concat([genres, numeriques_norm], axis=1)
X1.shape

(6942, 25)

> **Constat** : `X1` a 6 942 lignes (une par film) et 25 colonnes (21 genres + 4 colonnes chiffrées), toutes comprises entre 0 et 1.

> **Objectif** : recalculer la distance entre Alien et Aliens, cette fois avec les genres, pour voir leur poids face aux colonnes chiffrées.

In [82]:
i_alien = films[films["tconst"] == "tt0078748"].index[0]
i_aliens = films[films["tconst"] == "tt0090605"].index[0]
ecarts_X = X1.iloc[i_alien] - X1.iloc[i_aliens]
# On n'affiche que les colonnes où les deux films diffèrent
ecarts_X[ecarts_X != 0]

Action           -1.00000
Adventure        -1.00000
Sci-Fi            1.00000
startYear        -0.12500
runtimeMinutes   -0.17094
log_votes         0.03239
dtype: float64

In [83]:
print(f"Distance avec les genres : {np.sqrt((ecarts_X ** 2).sum()):.2f}")

Distance avec les genres : 1.75


> **Constat** : Alien et Aliens diffèrent sur 3 genres (Sci-Fi d'un côté, Action et Adventure de l'autre). Ces 3 genres apportent 3 à la somme des carrés, contre environ 0,05 pour les 4 colonnes chiffrées : la distance (1,75) est maintenant presque entièrement due aux genres. C'est l'inverse du problème de départ : ce sont désormais les genres qui dominent. On pourra rééquilibrer plus tard en donnant un poids à chaque bloc de colonnes.

> **Objectif** : trouver les 5 films les plus proches d'Alien. On utilise l'algorithme des k plus proches voisins (KNN), dans sa version la plus simple : `NearestNeighbors` de scikit-learn se contente de trouver les voisins, sans rien prédire (ni classification comme `KNeighborsClassifier`, ni régression comme `KNeighborsRegressor`). C'est de l'apprentissage non supervisé. La distance utilisée est la distance euclidienne sur les 25 colonnes de `X1`. On demande 6 voisins, car le plus proche d'Alien… est Alien lui-même (distance 0).

In [84]:
from sklearn.neighbors import NearestNeighbors

In [85]:
modele1 = NearestNeighbors(n_neighbors=6)
modele1.fit(X1)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Remarque** : dans scikit-learn, tous les outils ont une méthode `fit`, mais elle ne fait pas la même chose partout. Pour un modèle qui prédit (une régression linéaire, par exemple), `fit` *apprend* des paramètres à partir des données. Pour `NearestNeighbors`, il n'y a rien à apprendre : `fit` se contente de mémoriser les 6 942 lignes de `X1` et de les ranger pour accélérer la recherche (comme un annuaire classé par ordre alphabétique). Le vrai travail est fait plus tard, par `kneighbors`, qui calcule les distances au moment de la question. C'est pourquoi on dit que le KNN est un algorithme « paresseux » (*lazy learning*).

In [86]:
distances, indices = modele1.kneighbors(X1.iloc[[i_alien]])
indices, distances

(array([[ 352,  333,  596,  362,  384, 3084]]),
 array([[0.        , 0.37101037, 0.53099273, 0.54115017, 0.54537701,
         0.58106159]]))

> **Constat** : `kneighbors` renvoie deux tableaux : les positions des 6 films les plus proches (`indices`) et leurs distances (`distances`), du plus proche au plus éloigné. Le premier est Alien lui-même (position 352, distance 0). Les 5 autres sont ses voisins, à des distances de 0,37 à 0,58.

> **Objectif** : afficher les titres, années et genres des 6 films trouvés, avec leur distance à Alien.

In [87]:
voisins = films.iloc[indices[0]][["title", "startYear", "genres"]]
voisins["distance"] = distances[0]
voisins

,title,startYear,genres,distance
352,"Alien, le 8ème passager",1979,"Horror,Sci-Fi",0.000000
333,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.371010
596,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.530993
362,Chromosome 3,1979,"Horror,Sci-Fi",0.541150
384,Phantasm,1979,"Horror,Sci-Fi",0.545377
3084,28 semaines plus tard,2007,"Horror,Sci-Fi",0.581062


> **Constat** : les 5 voisins d'Alien ont exactement les mêmes genres (Horror, Sci-Fi), et 4 sur 5 sont sortis entre 1978 et 1983. Les suites d'Alien n'apparaissent pas : Aliens diffère d'Alien sur 3 genres, donc sa distance vaut au moins √3 ≈ 1,73, bien plus que ces voisins (0,37 à 0,58). Le moteur juge sur l'étiquette (genres, époque), pas sur le contenu : il ignore que les films de la saga racontent la même histoire. Il manque une description de ce dont parle le film : ce sera le rôle des mots-clés.

> **Objectif** : écrire des fonctions pour tester facilement le moteur sur n'importe quel film. `chercher(mot)` affiche les films dont le titre contient un mot, avec leur identifiant `tconst`. `recommander(tconst, X, modele)` affiche les 5 films les plus proches d'un film donné, pour une matrice `X` et un modèle donnés ; `recommander1(tconst)` l'applique à la version 1 (`X1` et `modele1`).

In [88]:
def chercher(mot):
    filtre = films["title"].str.contains(mot, case=False)
    return films[filtre][["tconst", "title", "startYear", "genres"]]

In [89]:
chercher("taxi")

,tconst,title,startYear,genres
265,tt0075314,Taxi Driver,1976,"Crime,Drama"
1984,tt0152930,Taxi,1998,"Action,Comedy,Crime"
2080,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2856,tt0410730,Taxidermie,2006,"Comedy,Horror"
3134,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
6096,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
6563,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [90]:
chercher("alien")

,tconst,title,startYear,genres
352,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
741,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1179,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1206,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1675,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2478,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
2687,tt0370263,Alien vs. Predator,2004,"Action,Adventure,Horror"
3220,tt0758730,Aliens vs Predator: Requiem,2007,"Action,Horror,Sci-Fi"
3397,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [91]:
chercher("matrix")

,tconst,title,startYear,genres
1919,tt0133093,Matrix,1999,"Action,Sci-Fi"
2187,tt0234215,Matrix Reloaded,2003,"Action,Sci-Fi"
2206,tt0242653,Matrix Revolutions,2003,"Action,Sci-Fi"
3679,tt10838180,Matrix Resurrections,2021,"Action,Sci-Fi"


> **Remarque** : *Matrix Resurrections* (2021), quatrième film de la saga, est bien dans le catalogue, alors que sa note IMDb est basse (5,6/10, source : imdb.com/title/tt10838180) : les nouvelles règles de sélection de l'équipe complètent les sagas.

> **Idée (à discuter en équipe)** : l'équipe a complété les sagas dans le catalogue (Matrix Resurrections, Alien vs. Predator…). Il reste à les montrer dans l'appli : aucune colonne d'IMDb ou de TMDB chargée ici n'indique directement qu'un film appartient à une saga.
>
> Piste simple : les sagas « cultes » sont peu nombreuses (quelques dizaines au plus). On peut les définir à la main dans un petit fichier `sagas.csv` (colonnes `saga` et `tconst`), et l'appli afficherait « Dans la même saga ».

> **Remarque** : `recommander` reçoit la matrice et le modèle en paramètres, au lieu d'aller chercher des variables du notebook. On pourra ainsi interroger plusieurs versions du moteur (`X1` et `modele1`, puis `X2` et `modele2`…) et les comparer côte à côte. Passer `X` en paramètre ne le copie pas : la fonction reçoit une référence vers le même tableau.

In [92]:
def recommander(tconst, X, modele):
    i = films[films["tconst"] == tconst].index[0]
    distances, indices = modele.kneighbors(X.iloc[[i]])
    voisins = films.iloc[indices[0][1:]][["tconst", "title", "startYear", "genres"]]
    voisins["distance"] = distances[0][1:]
    return voisins

In [93]:
def recommander1(tconst):
    return recommander(tconst, X1, modele1)

In [94]:
recommander1("tt0078748")

,tconst,title,startYear,genres,distance
333,tt0077745,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.371010
596,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.530993
362,tt0078908,Chromosome 3,1979,"Horror,Sci-Fi",0.541150
384,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.545377
3084,tt0463854,28 semaines plus tard,2007,"Horror,Sci-Fi",0.581062


In [95]:
recommander1("tt0077745")

,tconst,title,startYear,genres,distance
596,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.218314
362,tt0078908,Chromosome 3,1979,"Horror,Sci-Fi",0.236908
384,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.260370
296,tt0076590,Rage,1977,"Horror,Sci-Fi",0.303790
220,tt0073705,Frissons,1975,"Horror,Sci-Fi",0.318502


In [96]:
recommander1("tt0078908")

,tconst,title,startYear,genres,distance
384,tt0079714,Phantasm,1979,"Horror,Sci-Fi",0.042944
296,tt0076590,Rage,1977,"Horror,Sci-Fi",0.099910
220,tt0073705,Frissons,1975,"Horror,Sci-Fi",0.115201
596,tt0086491,La Quatrième Dimension,1983,"Horror,Sci-Fi",0.118909
756,tt0091083,From Beyond : Aux portes de l'au-delà,1986,"Horror,Sci-Fi",0.141354


> **Constat** : en enchaînant les recommandations (Alien → L'Invasion des profanateurs → Chromosome 3), on reste dans un petit groupe de films Horror, Sci-Fi de 1975–1986, et les distances diminuent (de 0,37–0,58 à 0,04–0,14) : on s'enfonce dans un groupe de films très proches les uns des autres, qui se recommandent entre eux. Ce moteur tourne un peu en rond, moins toutefois qu'un moteur plus précis : précision et diversité tirent en sens contraire. Ce point sera traité dans l'application (exclure les films déjà vus, mélanger proches et plus lointains).

In [97]:
chercher("taxi")

,tconst,title,startYear,genres
265,tt0075314,Taxi Driver,1976,"Crime,Drama"
1984,tt0152930,Taxi,1998,"Action,Comedy,Crime"
2080,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2856,tt0410730,Taxidermie,2006,"Comedy,Horror"
3134,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
6096,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
6563,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [98]:
recommander1("tt0152930")

,tconst,title,startYear,genres,distance
2067,tt0181316,Flic de haut vol,1999,"Action,Comedy,Crime",0.088886
2080,tt0183869,Taxi 2,2000,"Action,Comedy,Crime",0.096100
2355,tt0281364,La petite moutarde qui monte au nez,2001,"Action,Comedy,Crime",0.131723
2178,tt0227984,Le 51e état,2001,"Action,Comedy,Crime",0.139164
2300,tt0266915,Rush Hour 2,2001,"Action,Comedy,Crime",0.152148


> **Constat** : pour Taxi, Taxi 2 arrive deuxième, juste derrière Flic de haut vol (distances 0,089 et 0,096), parmi des comédies d'action policières de la même époque (1999–2001), toutes Action, Comedy, Crime. Le résultat est correct, car ces genres sont précis et partagés par la suite. Mais les distances sont très proches (0,09 à 0,15) : le moteur départage des films au même profil sur de petits écarts d'année ou de note. Il ignore ce qui fait l'identité de Taxi (film français, Marseille, voitures) : la plupart des voisins proposés sont anglophones.

> **Remarque** : ce premier moteur (21 genres et 4 colonnes chiffrées, distance euclidienne, sans poids) est appelé **version 1** dans la suite (`X1`, `modele1`, `recommander1`). La partie 5 construit une version 2, qui ajoute les mots-clés, des poids et la distance cosinus.

## 5. Le contenu des films : les mots-clés

> **Remarque** : dans cette partie, les cellules **Article** signalent les valeurs et les remarques reprises dans l'article « TF-IDF et similarité cosinus », qui accompagne le développement du moteur.

> **Objectif** : ajouter au moteur une description du contenu des films, grâce aux mots-clés TMDB (colonne `keywords`). On commence par regarder à quoi ressemblent ces mots-clés pour Alien et Aliens.

In [99]:
chercher("alien")

,tconst,title,startYear,genres
352,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
741,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1179,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1206,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1675,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2478,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
2687,tt0370263,Alien vs. Predator,2004,"Action,Adventure,Horror"
3220,tt0758730,Aliens vs Predator: Requiem,2007,"Action,Horror,Sci-Fi"
3397,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [100]:
filtre_deux = films["tconst"].isin(["tt0078748", "tt0090605"])
for texte in films[filtre_deux]["keywords"]:
    print(texte)
    print()

android, spacecraft, space marine, biology, dystopia, countdown, space suit, beheading, space travel, cowardice, alien, space, female protagonist, parasite, space opera, cosmos, xenomorph

android, space marine, extraterrestrial technology, spaceman, space travel, settler, colony, cryogenics, vacuum, space colony, warrior woman, alien, space, female protagonist, creature, desolate, female hero, xenomorph, desolate planet, strong female protagonist



In [101]:
mots_alien = films[films["tconst"] == "tt0078748"]["keywords"].iloc[0].split(", ")
mots_aliens = films[films["tconst"] == "tt0090605"]["keywords"].iloc[0].split(", ")
len(mots_alien), len(mots_aliens)

(17, 20)

In [102]:
communs = []
for mot in mots_alien:
    if mot in mots_aliens:
        communs.append(mot)
print("Nombre de mots clés commun :", len(communs))
communs

Nombre de mots clés commun : 7


['android',
 'space marine',
 'space travel',
 'alien',
 'space',
 'female protagonist',
 'xenomorph']

> **Constat** : Alien a 17 mots-clés, Aliens 20, et ils en partagent 7 : android, space marine, space travel, alien, space, female protagonist, xenomorph. Ces mots décrivent le contenu commun des deux films (une créature extraterrestre, l'espace, une héroïne), ce que les genres ne disaient pas.

> **Article** : section 1.1 (Alien et Aliens partagent sept mots-clés) et section 5.2 (les synonymes : « spacecraft » figure parmi les mots-clés d'Alien).

> **Objectif** : compter, pour chacun des 7 mots-clés communs, combien de films du catalogue le possèdent, pour voir s'ils sont rares ou fréquents.

In [103]:
listes_mots = films["keywords"].str.split(", ")

for mot in communs:
    nb = 0
    for liste in listes_mots:
        if mot in liste:
            nb = nb + 1
    print(mot, ":", nb, "films")

android : 37 films
space marine : 24 films
space travel : 41 films
alien : 112 films
space : 47 films
female protagonist : 72 films
xenomorph : 7 films


> **Constat** : les 7 mots communs n'ont pas la même rareté. « xenomorph » n'apparaît que dans 7 films du catalogue (sans doute la saga Alien), « space marine » dans 24, alors que « alien » apparaît dans 112 films et « female protagonist » dans 72. Partager un mot rare en dit beaucoup plus sur la ressemblance de deux films que partager un mot fréquent.

> **Article** : section 4.5 (colonne df du tableau des 7 mots communs).

In [104]:
for i in range(len(films)):
    if "xenomorph" in listes_mots.iloc[i]:
        print(films["title"].iloc[i], films["startYear"].iloc[i])

Alien, le 8ème passager 1979
Aliens, le retour 1986
Alien 3 1992
Alien, la résurrection 1997
Alien vs. Predator 2004
Aliens vs Predator: Requiem 2007
Alien: Covenant 2017


> **Constat** : les 7 films qui ont le mot-clé « xenomorph » sont bien des films de la saga Alien : les cinq films de la saga principale (1979, 1986, 1992, 1997, 2017) et les deux Alien vs. Predator (2004, 2007). Ce mot rare suffit presque à lui seul à identifier la saga. Mais Alien: Romulus (2024), présent dans le catalogue, n'apparaît pas : regardons ses mots-clés.

In [105]:
chercher("romulus")

,tconst,title,startYear,genres
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [106]:
films[films["tconst"] == "tt18412256"]["keywords"]

4860    alien, space
Name: keywords, dtype: str

> **Constat** : TMDB ne donne que deux mots-clés à Alien: Romulus (« alien, space »). Ce film récent était encore peu décrit par la communauté au moment de l'extraction. Avec si peu de mots-clés, il sera difficile au moteur de le rapprocher du reste de la saga.

> **Article** : section 5.2 (Ce que TF-IDF ne voit pas, la qualité des descriptions : Alien: Romulus n'a que deux mots-clés, « alien » et « space »).

### Le vocabulaire

> **Objectif** : compter les films et les mots-clés différents de tout le catalogue (la taille du vocabulaire). Le nombre de mots-clés différents est la dimension p de l'espace dans lequel chaque film deviendra un vecteur.

In [107]:
vocabulaire = set()
for liste in listes_mots:
    for mot in liste:
        vocabulaire.add(mot)

print("films :", len(films))
print("mots-clés différents :", len(vocabulaire))

films : 6942
mots-clés différents : 13340


> **Objectif** : compter les mots-clés de chaque film, pour savoir combien de ces coordonnées un film utilise réellement.

In [108]:
nb_mots = []
for liste in listes_mots:
    nb_mots.append(len(liste))
nb_mots = pd.Series(nb_mots)

print("minimum :", nb_mots.min())
print("moyenne :", round(nb_mots.mean(), 1))
print("maximum :", nb_mots.max())

minimum : 1
moyenne : 9.7
maximum : 75


> **Constat** : le catalogue compte n = 6 942 films, qui utilisent 13 340 mots-clés différents. Chaque film deviendra donc un vecteur de dimension p = 13 340, alors qu'il n'a qu'une dizaine de mots-clés en moyenne, et 75 au plus. Ces vecteurs sont presque entièrement remplis de 0 : on dit que la matrice des films est *creuse*.

> **Article** : section 2.4 (Dimension et creux) : n = 6 942 films, p = 13 340 mots-clés, et un film n'a que « quelques dizaines de mots-clés au plus ».

### La fréquence documentaire

> **Objectif** : calculer, pour chaque mot-clé, le nombre de films qui le contiennent : c'est sa *fréquence documentaire*, notée df. On parcourt `set(liste)` plutôt que `liste` : un `set` supprime les doublons, et un film ne doit compter qu'une fois par mot, même si ce mot est répété dans sa liste (voir la partie suivante).

In [109]:
compte = {}
for liste in listes_mots:
    for mot in set(liste):    # set : un film ne compte qu'une fois par mot
        compte[mot] = compte.get(mot, 0) + 1

print("xenomorph :", compte["xenomorph"])
print("alien :", compte["alien"])

xenomorph : 7
alien : 112


> **Constat** : on retrouve les valeurs obtenues plus haut avec une boucle mot par mot (7 films pour « xenomorph », 112 pour « alien ») : `compte[mot]` est bien la fréquence documentaire du mot.

> **Objectif** : compter les mots-clés présents dans un seul film (df = 1). Un tel mot ne peut rapprocher aucun film d'un autre.

In [110]:
nb_uniques = 0
for mot in compte:
    if compte[mot] == 1:
        nb_uniques = nb_uniques + 1

print("mots présents dans un seul film :", nb_uniques)
print("part du vocabulaire (%) :", round(nb_uniques / len(vocabulaire) * 100, 1))
print("mots restants :", len(vocabulaire) - nb_uniques)

mots présents dans un seul film : 6406
part du vocabulaire (%) : 48.0
mots restants : 6934


> **Constat** : 6 406 mots-clés, soit 48 % du vocabulaire, n'apparaissent que dans un seul film. Dans le produit scalaire de deux films, un mot ne compte que si les deux films l'ont : ces mots n'ajoutent donc rien à la ressemblance entre films. Pire, ils allongent le vecteur de leur film, ce qui l'éloigne un peu de tous les autres. En les retirant, il reste 6 934 mots-clés utiles.
>
> **Décision** : écarter les mots présents dans un seul film. Avec scikit-learn, c'est l'option `min_df=2` (voir plus bas).

> **Article** : section 2.4 (6 406 mots uniques, 48 %, 6 934 mots restants) et section 4.2 (remarque 1 : le cas df = 1).

### Les mots-clés répétés

> **Objectif** : compter les films dont la liste contient un mot-clé répété. Un `set` supprime les doublons : s'il est plus court que la liste, c'est qu'un mot y est répété.

In [111]:
nb_films = 0
for liste in listes_mots:
    if len(set(liste)) < len(liste):
        nb_films = nb_films + 1
nb_films

174

> **Objectif** : voir quels mots-clés sont le plus souvent répétés, et dans combien de films.

In [112]:
repetes = {}
for liste in listes_mots:
    for mot in set(liste):
        if liste.count(mot) > 1:
            repetes[mot] = repetes.get(mot, 0) + 1
pd.Series(repetes).sort_values(ascending=False).head(5)

new york city    30
england          20
france           17
california       15
japan             9
dtype: int64

> **Objectif** : afficher les films dans lesquels « england » est répété, avec le nombre de répétitions.

In [113]:
for i in range(len(films)):
    nb = listes_mots.iloc[i].count("england")
    if nb > 1:
        print(films["title"].iloc[i], ":", nb, "fois")

Les Chiens de paille : 2 fois
Vivre et laisser mourir : 2 fois
L'homme au pistolet d'or : 2 fois
Rien que pour vos yeux : 2 fois
Dangereusement vôtre : 2 fois
Tuer n'est pas jouer : 2 fois
Demain ne meurt jamais : 2 fois
Coup de foudre à Notting Hill : 2 fois
Le Journal de Bridget Jones : 2 fois
L'escorte : 2 fois
Miss Pettigrew : 2 fois
Auprès de moi toujours : 2 fois
RED 2 : 2 fois
Il était temps : 2 fois
Conjuring 2 : Le Cas Enfield : 2 fois
Le BGG : Le Bon Gros Géant : 2 fois
The Lady in the Van : 2 fois
Le Merveilleux Jardin secret de Bella Brown : 2 fois
6 Days : 2 fois
Juliet, Naked : 2 fois


> **Objectif** : chercher les mots-clés écrits plus de deux fois dans la liste d'un même film.

In [114]:
for i in range(len(films)):
    liste = listes_mots.iloc[i]
    for mot in set(liste):
        if liste.count(mot) > 2:
            print(films["title"].iloc[i], ":", mot, liste.count(mot), "fois")

Mystic River : massachusetts 3 fois
Il était une fois... : new york city 3 fois
Son of a Gun : australia 3 fois
Ocho apellidos vascos : spain 3 fois
Un jour de pluie à New York : new york city 3 fois


> **Objectif** : chercher les mots-clés présents dans un seul film, mais répétés dans sa liste. Ce sont eux qu'une boucle sans `set` compterait mal.

In [115]:
for i in range(len(films)):
    liste = listes_mots.iloc[i]
    for mot in set(liste):
        if compte[mot] == 1 and liste.count(mot) > 1:
            print(mot, ":", films["title"].iloc[i])

louisville : Le retour des morts-vivants


> **Constat** : 174 films ont au moins un mot-clé répété dans leur liste TMDB. Les mots les plus souvent répétés sont des lieux : « new york city » est répété dans 30 films, « england » dans 20 (à chaque fois exactement deux fois). Seuls 5 films ont un mot écrit trois fois, et aucun n'en a un écrit davantage. Ces répétitions sont des erreurs de saisie : elles ne disent rien de plus sur le film.
>
> **Remarque** : un de ces mots, « louisville », n'apparaît que dans un seul film. Sans le `set`, son compte vaudrait 2 et il échapperait au décompte des mots uniques : c'est pourquoi la boucle `compte` parcourt `set(liste)`.

> **Remarque** : les autres mots répétés sont partagés par de nombreux films. Par défaut, `TfidfVectorizer` compte les occurrences : la répétition doublerait le poids de ces mots dans les films concernés, et ces films paraîtraient artificiellement plus proches des autres films new-yorkais ou anglais.
>
> **Décision** : lors du calcul TF-IDF, utiliser l'option `binary=True` : un mot-clé présent compte 1, qu'il soit répété ou non.

> **Article** : annexe B, section B.2 (codage du tf : 174 films ont un mot-clé répété, par exemple « england » écrit deux fois).

### L'IDF : le poids de la rareté

> **Objectif** : calculer le poids IDF des 7 mots-clés communs à Alien et Aliens, avec la formule de l'article : idf(t) = ln(n / df(t)) + 1, où n est le nombre de films et df(t) le nombre de films qui contiennent le mot t. Plus un mot est rare, plus son poids est grand.

In [116]:
n = len(films)

def idf(mot):
    return np.log(n / compte[mot]) + 1

for mot in communs:
    print(f"{mot} : {compte[mot]} films, idf = {idf(mot):.2f}")

android : 37 films, idf = 6.23
space marine : 24 films, idf = 6.67
space travel : 41 films, idf = 6.13
alien : 112 films, idf = 5.13
space : 47 films, idf = 6.00
female protagonist : 72 films, idf = 5.57
xenomorph : 7 films, idf = 7.90


> **Objectif** : comparer le mot le plus rare (« xenomorph ») et le plus fréquent (« alien ») : combien de fois plus rare, combien de fois plus lourd ?

In [117]:
print("rapport des df :", round(compte["alien"] / compte["xenomorph"], 1))
print("rapport des idf :", round(idf("xenomorph") / idf("alien"), 2))

rapport des df : 16.0
rapport des idf : 1.54


> **Constat** : « xenomorph » (7 films) est seize fois plus rare que « alien » (112 films), mais son poids n'est que 1,54 fois plus grand (7,90 contre 5,13). C'est l'effet du logarithme : il favorise les mots rares sans écraser les autres.

> **Article** : section 4.2 (courbe de l'IDF, avec les points de « xenomorph », « space marine », « android » et « alien ») et section 4.5 (tableau des 7 mots communs, rapport 7,90 / 5,13 ≈ 1,54).

### Un exemple jouet

> **Objectif** : sur un petit exemple de 6 films (A à F), voir comment la pondération TF-IDF modifie les ressemblances. Les mots-clés sont inventés pour que chaque valeur se vérifie à la main ; A joue le rôle d'Alien.

In [118]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def decouper(texte):
    return texte.split(", ")    # un mot-clé = une expression entre virgules

jouet = ["xenomorph, spaceship, murder",    # A
         "xenomorph, android",              # B
         "murder, detective",               # C
         "murder, spaceship, princess",     # D
         "spaceship, android",              # E
         "murder, android"]                 # F

> **Objectif** : coder les 6 films en binaire (1 si le mot est présent, 0 sinon), puis calculer la similarité cosinus de A avec chaque film.

In [119]:
# Sans idf ni normalisation : la matrice des 0 et des 1
binaire = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                          binary=True, use_idf=False, norm=None)
X_jouet = binaire.fit_transform(jouet)
print(binaire.get_feature_names_out())
print(X_jouet.toarray().astype(int))

['android' 'detective' 'murder' 'princess' 'spaceship' 'xenomorph']
[[0 0 1 0 1 1]
 [1 0 0 0 0 1]
 [0 1 1 0 0 0]
 [0 0 1 1 1 0]
 [1 0 0 0 1 0]
 [1 0 1 0 0 0]]


In [120]:
# Cette matrice documents-termes peut-être affiché comme un DataFrame
pd.DataFrame(X_jouet.toarray(), columns=binaire.get_feature_names_out())

,android,detective,murder,princess,spaceship,xenomorph
0,0.0,0.0,1.0,0.0,1.0,1.0
1,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,1.0,1.0,0.0
4,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,1.0,0.0,0.0,0.0


In [121]:
cosine_similarity(X_jouet)[0].round(3)    # A, B, C, D, E, F

array([1.   , 0.408, 0.408, 0.667, 0.408, 0.408])

> **Constat** : en codage binaire, D est le plus proche de A (0,667) : ils partagent deux mots (murder, spaceship). Les quatre autres films partagent un seul mot avec A et sont à égalité (0,408), quel que soit ce mot.

> **Objectif** : refaire le calcul avec la pondération TF-IDF de l'article : idf non lissé (`smooth_idf=False`) et vecteurs non normalisés (`norm=None`).

In [122]:
tfidf_jouet = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                              smooth_idf=False, norm=None)
W_jouet = tfidf_jouet.fit_transform(jouet)
print(tfidf_jouet.get_feature_names_out())
print(tfidf_jouet.idf_.round(3))

['android' 'detective' 'murder' 'princess' 'spaceship' 'xenomorph']
[1.693 2.792 1.405 2.792 1.693 2.099]


In [123]:
cosine_similarity(W_jouet)[0].round(3)    # A, B, C, D, E, F

array([1.   , 0.537, 0.208, 0.448, 0.394, 0.295])

> **Constat** : le classement change complètement : B (0,537), D (0,448), E (0,394), F (0,295), C (0,208). Trois effets l'expliquent :
> - un mot partagé compte d'autant plus qu'il est rare : B (xenomorph, 2 films) passe devant F (murder, 4 films) ;
> - un mot qu'un film possède en propre l'éloigne, d'autant plus qu'il est rare : C et F partagent le même mot avec A, mais C a en plus « detective » (1 film) et F « android » (3 films) ;
> - un seul mot rare peut l'emporter sur deux mots courants : B passe devant D.
>
> En résumé, le codage binaire *compte* les mots partagés, la pondération TF-IDF les *pèse*.

> **Objectif** : comparer avec les réglages par défaut de scikit-learn : idf lissé, ln((1 + n) / (1 + df)) + 1, et vecteurs ramenés à une norme de 1.

In [124]:
defaut = TfidfVectorizer(tokenizer=decouper, token_pattern=None)
W_defaut = defaut.fit_transform(jouet)
print(defaut.idf_.round(3))
print(cosine_similarity(W_defaut)[0].round(3))    # A, B, C, D, E, F

[1.56  2.253 1.336 2.253 1.56  1.847]
[1.    0.511 0.247 0.501 0.399 0.315]


> **Constat** : le lissage fait comme si l'on ajoutait un film fictif contenant tous les mots. Les poids baissent, surtout ceux des mots rares (xenomorph passe de 2,099 à 1,847). Le classement reste le même (B, D, E, F, C), mais l'avance de B sur D se réduit (0,511 contre 0,501). La normalisation, elle, ne change aucun cosinus.

> **Article** : section 2.3 (la matrice des 6 films), section 3.3 (cosinus en codage binaire), section 4.4 (idf, cosinus TF-IDF et les trois constats) et annexe B, section B.2 (tableau sans lissage / avec lissage).

### TF-IDF sur le catalogue

> **Objectif** : appliquer TF-IDF aux mots-clés du catalogue, avec les deux décisions prises plus haut : `binary=True` (mots répétés) et `min_df=2` (mots présents dans un seul film). Les autres réglages restent ceux par défaut.

In [125]:
tfidf = TfidfVectorizer(tokenizer=decouper, token_pattern=None,
                        binary=True, min_df=2)
mots = tfidf.fit_transform(films["keywords"])
mots.shape

(6942, 6934)

> **Objectif** : compter les coefficients non nuls de la matrice. `mots` est une matrice creuse (*sparse*) : au lieu de stocker toutes ses cases, elle ne stocke que les cases non nulles, avec leur position. Leur nombre est donné par l'attribut `nnz` (*number of non-zeros*, « nombre de non-zéros »).

In [126]:
nb_cases = mots.shape[0] * mots.shape[1]
print("coefficients non nuls :", mots.nnz)
print("part des coefficients (%) :", round(mots.nnz / nb_cases * 100, 2))

coefficients non nuls : 60431
part des coefficients (%) : 0.13


> **Constat** : la matrice a 6 942 lignes (les films) et 6 934 colonnes : on retrouve exactement le nombre de mots restants calculé à la main. Elle ne contient que 60 431 coefficients non nuls, soit 0,13 % de ses cases.

> **Objectif** : comparer les poids IDF de scikit-learn (lissés) à ceux calculés à la main, pour les 7 mots communs à Alien et Aliens.

In [127]:
for mot in communs:
    j = tfidf.vocabulary_[mot]    # numéro de la colonne du mot
    print(f"{mot} : à la main {idf(mot):.2f}, scikit-learn {tfidf.idf_[j]:.2f}")

android : à la main 6.23, scikit-learn 6.21
space marine : à la main 6.67, scikit-learn 6.63
space travel : à la main 6.13, scikit-learn 6.11
alien : à la main 5.13, scikit-learn 5.12
space : à la main 6.00, scikit-learn 5.97
female protagonist : à la main 5.57, scikit-learn 5.56
xenomorph : à la main 7.90, scikit-learn 7.77


> **Constat** : sur le catalogue, le lissage ne change presque rien : les écarts sont de quelques centièmes, sauf pour « xenomorph » (7,90 contre 7,77), le plus rare. On peut donc garder le réglage par défaut.

> **Objectif** : voir pourquoi il faut donner à `TfidfVectorizer` notre propre fonction de découpage. Sans elle, scikit-learn découpe le texte en mots isolés.

In [128]:
mots_isoles = TfidfVectorizer()
mots_isoles.fit(films["keywords"])
print("space marine :", "space marine" in mots_isoles.vocabulary_)
print("marine :", "marine" in mots_isoles.vocabulary_)

space marine : False
marine : True


> **Constat** : avec le découpage par défaut, « space marine » n'existe plus comme terme : il est coupé en « space » et « marine », et l'expression perd son sens. La fonction `decouper` garde chaque mot-clé entier.

> **Remarque** : on ne nettoie pas le texte : pas de suppression des mots vides, pas de regroupement des formes d'un même mot (stemming ou lemmatisation). Ces traitements servent pour du texte libre, comme des avis ou des résumés. Nos mots-clés sont des étiquettes courtes, écrites en minuscules (« space marine », « xenomorph ») : le découpage aux virgules suffit.

> **Idée (à discuter en équipe)** : ajouter les résumés des films (colonne `overview` du fichier TMDB, absente du catalogue actuel) comme huitième bloc TF-IDF, avec son propre poids. Contrairement aux mots-clés, c'est du texte libre, en anglais : il faudrait alors le nettoyer (mots vides avec `stop_words="english"`, éventuellement lemmatisation ou stemming, en mesurant si ça aide) et régler `min_df`, `max_df`, voire `ngram_range=(1, 2)`. Atout : les résumés citent les personnages (Ripley, Neo, Astérix…), des mots rares qui relieraient les films d'une même saga. Coût : un texte plus bruité, et de nouveaux tests sur les films tests.

> **Article** : annexe B, section B.1 (code, taille de la matrice, 60 431 coefficients non nuls, 0,13 %) et section B.2 (découpage, `binary`, lissage, `min_df`).

### Modèle version 2

> **Objectif** : construire une nouvelle matrice `X2` en six blocs, chacun multiplié par son propre poids : les genres, chacune des 4 valeurs numériques normalisées (année, durée, note, popularité), et les mots-clés TF-IDF. On commence par les valeurs numériques : leurs 4 poids sont rangés dans un dictionnaire (0,7 pour toutes, pour commencer).

In [129]:
poids_num = {"startYear": 0.7, "runtimeMinutes": 0.7,
             "averageRating": 0.7, "log_votes": 0.7}

num_ponderes = numeriques_norm.copy()
for col in poids_num:
    num_ponderes[col] = numeriques_norm[col] * poids_num[col]
num_ponderes.head()

,startYear,runtimeMinutes,averageRating,log_votes
0,0.3875,0.329060,0.429333,0.365480
1,0.0000,0.179487,0.513333,0.158644
2,0.0000,0.221368,0.494667,0.170144
3,0.0000,0.317094,0.485333,0.266763
4,0.0000,0.197436,0.494667,0.248985


In [130]:
num_ponderes.describe()

,startYear,runtimeMinutes,averageRating,log_votes
count,6942.000000,6942.000000,6942.000000,6942.000000
mean,0.450614,0.280622,0.465406,0.304424
std,0.174294,0.117604,0.060058,0.131144
min,0.000000,0.000000,0.000000,0.000000
25%,0.337500,0.197436,0.420000,0.203890
50%,0.487500,0.257265,0.457333,0.295174
75%,0.587500,0.335043,0.504000,0.399948
max,0.700000,0.700000,0.700000,0.700000


> **Constat** : chaque colonne numérique va maintenant de 0 à 0,7 au lieu de 0 à 1. Les valeurs sont multipliées par le même nombre, donc l'ordre des films ne change pas dans chaque colonne ; seul change le poids de la colonne dans les distances.

> **Objectif** : assembler la matrice `X2` des six blocs : les genres multipliés par 0,7, les 4 colonnes numériques pondérées, et les mots-clés TF-IDF (poids 1). `mots` est une matrice creuse : `toarray()` la transforme en tableau ordinaire pour pouvoir la coller aux autres blocs.

In [131]:
X2 = np.hstack([0.7 * genres.values,
                num_ponderes.values,
                mots.toarray()])
X2 = pd.DataFrame(X2)
X2.shape

(6942, 6959)

> **Constat** : `X2` a 6 942 lignes et 6 959 colonnes : 21 genres, 4 colonnes numériques et 6 934 mots-clés.

> **Objectif** : vérifier qu'aucun bloc n'écrase les autres. Le bloc mots-clés a 6 934 colonnes, contre 21 pour les genres et 4 pour les valeurs numériques : pèse-t-il beaucoup plus lourd dans la distance ? On calcule, pour chaque film, la longueur (la norme) de chaque bloc, c'est-à-dire la racine de la somme des carrés de ses valeurs.

In [132]:
blocs = {"genres": X2.iloc[:, :21],
         "numériques": X2.iloc[:, 21:25],
         "mots-clés": X2.iloc[:, 25:]}

longueurs = {}
for nom in blocs:
    longueurs[nom] = np.sqrt((blocs[nom] ** 2).sum(axis=1))
longueurs = pd.DataFrame(longueurs)
longueurs.describe().round(2)

,genres,numériques,mots-clés
count,6942.00,6942.00,6942.00
mean,1.14,0.80,1.00
std,0.13,0.14,0.06
min,0.70,0.42,0.00
25%,0.99,0.71,1.00
50%,1.21,0.80,1.00
75%,1.21,0.89,1.00
max,1.21,1.28,1.00


> **Constat** : malgré ses milliers de colonnes, le bloc mots-clés a une longueur de 1 pour presque tous les films (ligne `25%` : déjà 1) : `TfidfVectorizer` divise chaque ligne par sa norme (réglage `norm="l2"`, par défaut). Les genres ont une longueur de 1,14 en moyenne (0,7 multiplié par la racine du nombre de genres du film), les valeurs numériques de 0,80. Les trois blocs sont du même ordre de grandeur : aucun n'écrase les autres, et ce sont les poids qui règlent leur importance.
>
> **Remarque** : le cours sur la similarité conseille de chercher les voisins sur le texte seul, car des milliers de colonnes TF-IDF « pèseraient presque toute la distance ». Ce n'est pas le cas ici, grâce à la normalisation de chaque ligne : le nombre de colonnes d'un bloc ne compte pas, seule sa longueur compte.

> **Objectif** : la ligne `min` montre qu'au moins un film a un bloc mots-clés de longueur 0. Compter ces films.

In [133]:
(longueurs["mots-clés"] == 0).sum()

np.int64(29)

> **Constat** : 29 films ont un bloc mots-clés entièrement nul : chacun de leurs mots-clés n'apparaît que dans ce film, et `min_df=2` les a tous retirés. Ces films ne sont comparés aux autres que par leurs genres et leurs valeurs numériques.

> **Objectif** : créer un nouveau modèle KNN sur `X2`, cette fois avec la distance cosinus (`metric="cosine"`) : elle compare la direction des vecteurs, pas leur longueur. Un film décrit par beaucoup de mots-clés n'est donc pas éloigné des autres pour cette seule raison.

In [134]:
modele2 = NearestNeighbors(n_neighbors=6, metric="cosine")
modele2.fit(X2)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Objectif** : définir `recommander2`, qui interroge la version 2 grâce à la fonction générique `recommander` de la partie 4, puis une fonction `comparer` qui affiche côte à côte les titres recommandés par les deux versions. On ne compare pas les distances : la version 1 utilise la distance euclidienne, la version 2 la distance cosinus, leurs valeurs ne sont pas sur la même échelle.

In [135]:
def recommander2(tconst):
    return recommander(tconst, X2, modele2)

In [136]:
def comparer(tconst):
    v1 = recommander1(tconst)
    v2 = recommander2(tconst)
    return pd.DataFrame({"version 1": v1["title"].values,
                         "version 2": v2["title"].values})

> **Objectif** : comparer les versions 1 et 2 sur Alien. Dans la version 1, les suites d'Alien n'apparaissaient pas parmi ses voisins.

In [137]:
chercher("Alien")

,tconst,title,startYear,genres
352,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
741,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1179,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1206,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1675,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2478,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
2687,tt0370263,Alien vs. Predator,2004,"Action,Adventure,Horror"
3220,tt0758730,Aliens vs Predator: Requiem,2007,"Action,Horror,Sci-Fi"
3397,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [138]:
comparer("tt0078748")    # Alien

,version 1,version 2
0,L'Invasion des profanateurs,Alien 3
1,La Quatrième Dimension,Alien: Covenant
2,Chromosome 3,Alien: Romulus
3,Phantasm,L'Invasion des profanateurs
4,28 semaines plus tard,La Chose


> **Constat** : la version 2 retrouve trois films de la saga (Alien 3, Alien: Covenant, Alien: Romulus), absents de la version 1, puis L'Invasion des profanateurs et La Chose, un autre huis clos avec une créature venue d'ailleurs. La version 1 ne proposait que des films aux mêmes genres (Horror, Sci-Fi) et de la même époque. Alien: Romulus arrive troisième alors qu'il n'a que deux mots-clés (« alien » et « space ») : Alien possède ces deux mots, et la distance cosinus ne pénalise pas un film peu décrit, puisqu'elle ignore la longueur des vecteurs. En revanche, Aliens n'apparaît toujours pas : il diffère d'Alien sur trois genres (Action et Adventure d'un côté, Sci-Fi de l'autre).

> **Objectif** : creuser la saga Alien. Deux films de la saga principale manquent dans le top 5 de la version 2 (Aliens et Alien, la résurrection) : on cherche leur rang parmi les 100 plus proches voisins d'Alien, avec leurs genres et leur distance.

In [139]:
i_alien = films[films["tconst"] == "tt0078748"].index[0]
distances, indices = modele2.kneighbors(X2.iloc[[i_alien]], n_neighbors=100)
for rang in range(100):
    j = indices[0][rang]
    if "Alien" in films["title"].iloc[j]:
        print(rang, films["title"].iloc[j], films["genres"].iloc[j], round(distances[0][rang], 3))

0 Alien, le 8ème passager Horror,Sci-Fi 0.0
1 Alien 3 Action,Horror,Sci-Fi 0.334
2 Alien: Covenant Horror,Sci-Fi,Thriller 0.355
3 Alien: Romulus Horror,Sci-Fi,Thriller 0.358
6 Alien, la résurrection Action,Horror,Sci-Fi 0.376
44 Aliens vs Predator: Requiem Action,Horror,Sci-Fi 0.439
71 Aliens, le retour Action,Adventure,Horror 0.455


> **Constat** : Alien, la résurrection est 6ᵉ, juste hors du top 5 : ses genres conviennent, mais il partage moins de mots-clés avec Alien. Aliens n'est que 71ᵉ, alors qu'il partage 7 mots-clés avec Alien, dont « xenomorph » (voir le début de la partie 5). La raison est dans ses genres : IMDb le classe en Action, Adventure, Horror, sans Sci-Fi. Il ne partage donc qu'un genre avec Alien (Horror, Sci-Fi) et en diffère sur trois, ce qui suffit, avec un poids de 0,7, à le repousser loin. Aliens vs Predator: Requiem, qui a aussi « xenomorph », n'est que 44ᵉ. Les genres pèsent encore beaucoup ; un poids plus faible pour les genres sera à essayer lors du réglage des poids.

> **Objectif** : comparer les versions 1 et 2 sur Taxi. Dans la version 1, Taxi 2 arrivait en tête.

In [140]:
chercher("Taxi")

,tconst,title,startYear,genres
265,tt0075314,Taxi Driver,1976,"Crime,Drama"
1984,tt0152930,Taxi,1998,"Action,Comedy,Crime"
2080,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2856,tt0410730,Taxidermie,2006,"Comedy,Horror"
3134,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
6096,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
6563,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [141]:
comparer("tt0152930")    # Taxi

,version 1,version 2
0,Flic de haut vol,30 minutes maximum
1,Taxi 2,Mise à l'épreuve
2,La petite moutarde qui monte au nez,2 Guns
3,Le 51e état,Flic de haut vol
4,Rush Hour 2,Cops


> **Objectif** : comprendre pourquoi Taxi 2 disparaît : afficher les mots-clés de Taxi et de Taxi 2, et ceux qu'ils partagent.

In [142]:
mots_taxi = films[films["tconst"] == "tt0152930"]["keywords"].iloc[0].split(", ")
mots_taxi2 = films[films["tconst"] == "tt0183869"]["keywords"].iloc[0].split(", ")
print("Taxi :", mots_taxi)
print("Taxi 2 :", mots_taxi2)

communs_taxi = []
for mot in mots_taxi:
    if mot in mots_taxi2:
        communs_taxi.append(mot)
print("En commun :", communs_taxi)

Taxi : ['taxi', 'taxi driver', 'marseille', 'france', 'pizza boy', 'pizza delivery', 'police officer', 'bank robbery']
Taxi 2 : ['paris', 'france', 'yakuza', 'police operation', 'special unit']
En commun : ['france']


> **Constat** : en version 2, Taxi 2 disparaît des recommandations de Taxi, au profit de comédies policières américaines (30 minutes maximum, 2 Guns…). Les deux films ne partagent qu'un mot-clé, « france » : les mots-clés de Taxi 2 décrivent son intrigue, pas ce qui le relie au premier film. Les films américains partagent probablement avec Taxi des mots-clés plus précis, liés à son intrigue (livraison de pizzas, braquage, police). Rien, dans la matrice X2, n'indique que Taxi est un film français : c'est un défaut de la version 2.

### Modèle version 3

> **Objectif** : ajouter un septième bloc, la langue originale du film (colonne `original_language`, déjà présente dans le catalogue), pour que le moteur sache par exemple que Taxi est un film français. Chaque langue devient une colonne 0/1 : un film n'a qu'une langue, c'est donc du *one-hot* classique (contrairement aux genres, en *multi-hot*).

In [143]:
langues = pd.get_dummies(films["original_language"].fillna("inconnue")).astype(int)
langues.shape

(6942, 50)

> **Constat** : le catalogue compte 50 langues originales différentes : le bloc langue ajoute donc 50 colonnes, une par langue. Chaque ligne contient un seul 1, dans la colonne de la langue du film.

> **Objectif** : voir comment les films se répartissent entre les langues.

In [144]:
films["original_language"].value_counts().head(10)

original_language
en    4971
fr     739
hi     156
ja     149
es      97
ko      67
it      66
cn      62
ta      61
de      57
Name: count, dtype: int64

> **Constat** : les langues sont très inégalement représentées : 4 971 films sur 6 942, près des trois quarts du catalogue, sont en anglais (en). Vient ensuite le français (739 films), loin devant l'hindi (156) et le japonais (149). La colonne « en » ne distinguera donc pas les films anglophones entre eux, mais le bloc langue devrait aider les films des autres langues, comme Taxi, à retrouver des films de leur langue.

> **Objectif** : assembler la matrice `X3` : les six blocs de la version 2, plus la langue avec un poids de 0,8. On crée ensuite le modèle et la fonction `recommander3`, comme pour la version 2.

In [145]:
X3 = np.hstack([0.7 * genres.values,
                num_ponderes.values,
                mots.toarray(),
                0.8 * langues.values])
X3 = pd.DataFrame(X3)
X3.shape

(6942, 7009)

In [146]:
modele3 = NearestNeighbors(n_neighbors=6, metric="cosine")
modele3.fit(X3)

def recommander3(tconst):
    return recommander(tconst, X3, modele3)

> **Objectif** : réécrire `comparer` pour afficher côte à côte les recommandations des trois versions, puis retester Taxi.

In [147]:
def comparer(tconst):
    v1 = recommander1(tconst)
    v2 = recommander2(tconst)
    v3 = recommander3(tconst)
    return pd.DataFrame({"version 1": v1["title"].values,
                         "version 2": v2["title"].values,
                         "version 3": v3["title"].values})

In [148]:
chercher("Taxi")

,tconst,title,startYear,genres
265,tt0075314,Taxi Driver,1976,"Crime,Drama"
1984,tt0152930,Taxi,1998,"Action,Comedy,Crime"
2080,tt0183869,Taxi 2,2000,"Action,Comedy,Crime"
2856,tt0410730,Taxidermie,2006,"Comedy,Horror"
3134,tt0476884,Taxi 9211,2006,"Comedy,Drama,Thriller"
6096,tt4359416,Taxi Téhéran,2015,"Comedy,Drama"
6563,tt6878038,A Taxi Driver,2017,"Action,Drama,History"


In [149]:
comparer("tt0152930")

,version 1,version 2,version 3
0,Flic de haut vol,30 minutes maximum,Taxi 2
1,Taxi 2,Mise à l'épreuve,Dobermann
2,La petite moutarde qui monte au nez,2 Guns,Micmacs à tire-larigot
3,Le 51e état,Flic de haut vol,La petite moutarde qui monte au nez
4,Rush Hour 2,Cops,"Bon Cop, Bad Cop"


> **Constat** : avec la langue, Taxi 2 revient en tête, suivi de films en français : Dobermann, Micmacs à tire-larigot, La petite moutarde qui monte au nez, et Bon Cop, Bad Cop (film québécois en partie francophone). Le reste de la saga n'apparaît pas, mais `chercher("Taxi")` montre qu'il n'est pas dans le catalogue, malgré les nouvelles règles de sélection : Taxi 3 (note IMDb 5,8), Taxi 4 (5,6) et Taxi 5 (4,6) n'y sont pas (source : imdb.com). La version 3 retrouve donc toute la saga disponible.

In [150]:
chercher("Alien")

,tconst,title,startYear,genres
352,tt0078748,"Alien, le 8ème passager",1979,"Horror,Sci-Fi"
741,tt0090605,"Aliens, le retour",1986,"Action,Adventure,Horror"
1179,tt0102744,Mr Quigley l'Australien,1990,"Action,Adventure,Drama"
1206,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi"
1675,tt0118583,"Alien, la résurrection",1997,"Action,Horror,Sci-Fi"
2478,tt0317740,Braquage à l'italienne,2003,"Action,Crime,Thriller"
2687,tt0370263,Alien vs. Predator,2004,"Action,Adventure,Horror"
3220,tt0758730,Aliens vs Predator: Requiem,2007,"Action,Horror,Sci-Fi"
3397,tt0892782,Monstres contre Aliens,2009,"Action,Adventure,Animation"
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller"


In [151]:
comparer("tt0078748")

,version 1,version 2,version 3
0,L'Invasion des profanateurs,Alien 3,Alien 3
1,La Quatrième Dimension,Alien: Covenant,Alien: Covenant
2,Chromosome 3,Alien: Romulus,Alien: Romulus
3,Phantasm,L'Invasion des profanateurs,L'Invasion des profanateurs
4,28 semaines plus tard,La Chose,"Alien, la résurrection"


In [152]:
recommander2("tt0078748")

,tconst,title,startYear,genres,distance
1206,tt0103644,Alien 3,1992,"Action,Horror,Sci-Fi",0.334172
5234,tt2316204,Alien: Covenant,2017,"Horror,Sci-Fi,Thriller",0.355003
4860,tt18412256,Alien: Romulus,2024,"Horror,Sci-Fi,Thriller",0.357645
333,tt0077745,L'Invasion des profanateurs,1978,"Horror,Sci-Fi",0.372632
544,tt0084787,La Chose,1982,"Horror,Mystery,Sci-Fi",0.375079


> **Constat** : pour Alien, la version 3 reprend les quatre premiers films de la version 2, mais remplace La Chose par Alien, la résurrection : elle propose ainsi quatre films de la saga. Ces films sont tous en anglais, comme les trois quarts du catalogue : la colonne « en » vaut 1 pour tous et ne les distingue pas entre eux. Le changement vient d'un très petit écart : en version 2, Alien, la résurrection (6ᵉ, distance 0,376) suivait La Chose (5ᵉ, 0,375) de très près. La langue aide les films non anglophones, comme Taxi, sans dégrader les autres.

> **Objectif** : comparer les trois versions sur six autres films tests, connus de tous et de genres variés : Toy Story, La Communauté de l'anneau, Matrix (anglophones), puis Astérix & Obélix : Mission Cléopâtre, Intouchables et Amélie (francophones).

In [153]:
chercher("toy")

,tconst,title,startYear,genres
1559,tt0114709,Toy Story,1995,"Adventure,Animation,Comedy"
1791,tt0120363,Toy Story 2,1999,"Adventure,Animation,Comedy"
2964,tt0435761,Toy Story 3,2010,"Adventure,Animation,Comedy"
4963,tt1979376,Toy Story 4,2019,"Adventure,Animation,Comedy"
5595,tt29355505,Toy Story 5,2026,"Adventure,Animation,Comedy"
6128,tt4562518,Citoyen d'honneur,2016,"Comedy,Drama"


> **Constat** : la version 1 propose des films d'animation de la même époque (Monstres & Cie, Shrek, Aladdin…), avec Toy Story 2 en 2ᵉ position. Les versions 2 et 3 placent en tête trois suites (Toy Story 4, 3 et 2), puis deux autres films Pixar (Monstres & Cie, Vice-versa 2). Nette amélioration ; la langue ne change rien, tous ces films étant anglophones. Toy Story 5 (2026), présent dans le catalogue, n'apparaît dans aucune version.

In [154]:
comparer("tt0114709")

,version 1,version 2,version 3
0,Monstres & Cie,Toy Story 4,Toy Story 4
1,Toy Story 2,Toy Story 3,Toy Story 3
2,Shrek,Toy Story 2,Toy Story 2
3,Aladdin,Monstres & Cie,Monstres & Cie
4,L'Âge de glace,Vice-versa 2,Vice-versa 2


In [155]:
chercher("anneau")

,tconst,title,startYear,genres
334,tt0077869,Le Seigneur des anneaux,1978,"Adventure,Animation,Fantasy"
1826,tt0120737,Le Seigneur des anneaux : La Communauté de l'a...,2001,"Adventure,Drama,Fantasy"
2037,tt0167260,Le Seigneur des anneaux : Le Retour du roi,2003,"Adventure,Drama,Fantasy"
2038,tt0167261,Le Seigneur des anneaux : Les Deux Tours,2002,"Adventure,Drama,Fantasy"
4400,tt14824600,Le Seigneur des Anneaux : La Guerre des Rohirrim,2024,"Action,Adventure,Animation"
6229,tt5027774,3 Billboards : Les Panneaux de la vengeance,2017,"Comedy,Crime,Drama"
6887,tt9376612,Shang-Chi et la Légende des Dix Anneaux,2021,"Action,Adventure,Fantasy"


In [156]:
comparer("tt0120737")

,version 1,version 2,version 3
0,Le Seigneur des anneaux : Les Deux Tours,Le Seigneur des anneaux : Le Retour du roi,Le Seigneur des anneaux : Le Retour du roi
1,Le Seigneur des anneaux : Le Retour du roi,Le Seigneur des anneaux : Les Deux Tours,Le Seigneur des anneaux : Les Deux Tours
2,Big Fish,Black Is King,Black Is King
3,The Fall: Le voyage imaginaire,Le Hobbit : La Désolation de Smaug,Le Hobbit : La Désolation de Smaug
4,Excalibur,Le Hobbit: La Bataille des Cinq Armées,Le Hobbit: La Bataille des Cinq Armées


> **Constat** : les trois versions retrouvent les deux suites. Les versions 2 et 3 remplacent Big Fish, The Fall et Excalibur, trois films de fantasy sans lien avec la saga, par les deux films Le Hobbit, qui se déroulent dans le même univers ; mais Black Is King, un album visuel sans rapport, s'intercale en 3ᵉ position. Amélioration ; versions 2 et 3 identiques.

In [157]:
chercher("matrix")

,tconst,title,startYear,genres
1919,tt0133093,Matrix,1999,"Action,Sci-Fi"
2187,tt0234215,Matrix Reloaded,2003,"Action,Sci-Fi"
2206,tt0242653,Matrix Revolutions,2003,"Action,Sci-Fi"
3679,tt10838180,Matrix Resurrections,2021,"Action,Sci-Fi"


In [158]:
comparer("tt0133093")

,version 1,version 2,version 3
0,Avengers,Matrix Reloaded,Matrix Reloaded
1,Matrix Reloaded,Matrix Resurrections,Matrix Resurrections
2,X-Men : Le Commencement,Matrix Revolutions,Matrix Revolutions
3,Matrix Revolutions,Terminator 3 : Le Soulèvement des machines,Terminator 3 : Le Soulèvement des machines
4,Iron Man 2,Avengers,Avengers


> **Constat** : la version 1 propose surtout des films de super-héros (Avengers, X-Men, Iron Man 2), avec Matrix Reloaded en 2ᵉ position et Matrix Revolutions en 4ᵉ. Les versions 2 et 3 placent en tête les trois suites, Matrix Reloaded, Matrix Resurrections et Matrix Revolutions. Amélioration ; versions 2 et 3 identiques.

In [159]:
chercher("astérix")

,tconst,title,startYear,genres
195,tt0072901,Les 12 Travaux d'Astérix,1976,"Adventure,Animation,Comedy"
667,tt0088748,Astérix et la Surprise de César,1985,"Adventure,Animation,Comedy"
744,tt0090667,Astérix chez les Bretons,1986,"Adventure,Animation,Comedy"
970,tt0096842,Astérix et le Coup du menhir,1989,"Adventure,Animation,Comedy"
1923,tt0133385,Astérix & Obélix contre César,1999,"Adventure,Comedy,Family"
2233,tt0250223,Astérix & Obélix : Au service de Cléopâtre,2002,"Adventure,Comedy,Family"
2690,tt0371552,Astérix et les Vikings,2006,"Adventure,Animation,Comedy"
5961,tt3759416,Astérix : Le Domaine des dieux,2014,"Adventure,Animation,Comedy"
6728,tt8001346,Astérix: Le Secret de la potion magique,2018,"Adventure,Animation,Comedy"


In [160]:
comparer("tt0250223")

,version 1,version 2,version 3
0,Astérix & Obélix contre César,Paddington 2,Astérix & Obélix contre César
1,Le petit monde de Charlotte,Aladdin,Le Petit Nicolas
2,Histoires enchantées,Astérix & Obélix contre César,Les vacances du petit Nicolas
3,Major Payne,Paddington,La Guerre des boutons
4,Les Désastreuses Aventures des orphelins Baude...,Le Retour de Mary Poppins,La nouvelle guerre des boutons


> **Constat** : la version 1 place en tête Astérix & Obélix contre César, l'autre film en prises de vue réelles, suivi de comédies familiales anglophones. La version 2 le relègue en 3ᵉ position, parmi des films familiaux anglophones (Paddington, Aladdin…). La version 3 le remet en tête, suivi de classiques familiaux français (Le Petit Nicolas, Les vacances du petit Nicolas, La Guerre des boutons, La nouvelle guerre des boutons). Aucune version ne propose les dessins animés de la saga : 1 seul des 8 autres films Astérix est retrouvé. Les dessins animés ont sans doute des genres (Animation) et des mots-clés trop différents de ceux d'un film en prises de vue réelles. Un fichier de sagas fait main réglerait ce cas.

In [161]:
chercher("Intouchables")

,tconst,title,startYear,genres
4697,tt1675434,Intouchables,2011,"Comedy,Drama"


In [162]:
comparer("tt1675434")

,version 1,version 2,version 3
0,The Grand Budapest Hotel,Dasvi,Le grand bain
1,Birdman ou (La Surprenante vertu de l'ignorance),Dog,Simple comme Sylvain
2,Little Miss Sunshine,El buen patrón,Demain tout commence
3,Lost in Translation,The Upside - Seconde Chance,C.R.A.Z.Y.
4,Juno,Admis à tout prix,Monsieur Ibrahim et les fleurs du Coran


> **Constat** : la version 1 propose des comédies dramatiques anglophones (The Grand Budapest Hotel, Birdman…). La version 2 mélange des films de pays variés (Dasvi, El buen patrón…) et retrouve The Upside, le remake américain d'Intouchables. La version 3 ne propose plus que des films francophones (Le grand bain, Simple comme Sylvain, Demain tout commence, C.R.A.Z.Y., Monsieur Ibrahim et les fleurs du Coran), mais elle perd The Upside. Résultat mitigé : la langue rapproche des films par leur langue, pas forcément par leur histoire.

In [163]:
chercher("Amélie")

,tconst,title,startYear,genres
2144,tt0211915,Amélie des Abbesses,2001,"Comedy,Romance"


In [164]:
comparer("tt0211915")

,version 1,version 2,version 3
0,"40 ans, toujours puceau",Un accord parfait,Paris-Manhattan
1,Mary à tout prix,Paris-Manhattan,L'arnacoeur
2,Serial noceurs,Admis à tout prix,Simple comme Sylvain
3,Mon beau-père et moi,Greenberg,Lol
4,"En cloque, mode d'emploi","Hitch, expert en séduction",Amour & Turbulences


> **Constat** : la version 1 propose des comédies romantiques américaines (Mary à tout prix, 40 ans, toujours puceau…) ; la version 2 surtout des films anglophones, avec un film français (Paris-Manhattan). La version 3 ne propose que des films francophones (Paris-Manhattan, L'arnacoeur, Simple comme Sylvain, Lol, Amour & Turbulences), plus proches du public d'Amélie, mais rarement de son univers poétique.
>
> **Remarque** : dans le catalogue, le titre français d'Amélie est « Amélie des Abbesses » au lieu de « Le Fabuleux Destin d'Amélie Poulain » : le choix du titre français est à revoir avec l'équipe.

### Bilan des trois versions

> **Constat** : sur les huit films tests, la comparaison des trois versions donne :
>
> | Film | Version 1 | Version 2 | Version 3 |
> |---|---|---|---|
> | Alien | aucune suite | 3 films de la saga | 4 films de la saga |
> | Taxi | Taxi 2 en 2ᵉ position | Taxi 2 disparaît | Taxi 2 en tête, films français |
> | Toy Story | Toy Story 2 seulement | 3 suites | 3 suites |
> | La Communauté de l'anneau | 2 suites, fantasy | 2 suites, 2 Hobbit | 2 suites, 2 Hobbit |
> | Matrix | 2 suites, super-héros | les 3 suites en tête | les 3 suites en tête |
> | Intouchables | comédies anglophones | dont son remake | films francophones |
> | Amélie | comédies américaines | films surtout anglophones | films francophones |
> | Mission Cléopâtre | 1 Astérix en tête | 1 Astérix (3ᵉ) | 1 Astérix en tête, films français |
>
> Les mots-clés (version 2) font retrouver les sagas et les univers, ce que les genres seuls ne permettaient pas. La langue (version 3) corrige les films non anglophones, comme Taxi ou Mission Cléopâtre, sans dégrader les films anglophones ; mais elle peut aussi rapprocher des films seulement parce qu'ils partagent une langue (Intouchables perd son remake).
>
> **Décision** : retenir la version 3 comme base du moteur.

### Réglage des poids

> **Objectif** : tester l'importance de la durée. On reprend la version 3 en mettant le poids de la durée à 0, sans changer les autres poids, puis on compare les recommandations avec celles de la version 3.

In [165]:
poids_num

{'startYear': 0.7,
 'runtimeMinutes': 0.7,
 'averageRating': 0.7,
 'log_votes': 0.7}

In [166]:
poids_sans_duree = poids_num.copy()
poids_sans_duree["runtimeMinutes"] = 0

num_sans_duree = numeriques_norm.copy()
for col in poids_sans_duree:
    num_sans_duree[col] = numeriques_norm[col] * poids_sans_duree[col]

X3_sans_duree = pd.DataFrame(np.hstack([0.7 * genres.values,
                                        num_sans_duree.values,
                                        mots.toarray(),
                                        0.8 * langues.values]))
modele3_sans_duree = NearestNeighbors(n_neighbors=6, metric="cosine")
modele3_sans_duree.fit(X3_sans_duree)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",6
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


> **Objectif** : comparer, sur les huit films tests, les recommandations de la version 3 et de la variante sans durée.

In [167]:
def recommander3_sans_duree(tconst):
    return recommander(tconst, X3_sans_duree, modele3_sans_duree)

def comparer_duree(tconst):
    v3 = recommander3(tconst)
    v3_sans = recommander3_sans_duree(tconst)
    return pd.DataFrame({"version 3": v3["title"].values,
                         "sans durée": v3_sans["title"].values})

> **Remarque** : dans une cellule, seul le résultat de la dernière ligne s'affiche automatiquement en tableau. Dans une boucle, il faut donc appeler soi-même `display`, une fonction de Jupyter disponible sans import, pour afficher chaque DataFrame sous sa forme habituelle ; `print` l'afficherait en texte brut.

In [168]:
films_tests = {"Alien": "tt0078748", "Taxi": "tt0152930",
               "Toy Story": "tt0114709", "La Communauté de l'anneau": "tt0120737",
               "Matrix": "tt0133093", "Intouchables": "tt1675434",
               "Amélie": "tt0211915", "Mission Cléopâtre": "tt0250223"}

for titre in films_tests:
    print(titre)
    display(comparer_duree(films_tests[titre]))

Alien


,version 3,sans durée
0,Alien 3,Alien 3
1,Alien: Covenant,Alien: Covenant
2,Alien: Romulus,Alien: Romulus
3,L'Invasion des profanateurs,L'Invasion des profanateurs
4,"Alien, la résurrection","Alien, la résurrection"


Taxi


,version 3,sans durée
0,Taxi 2,Taxi 2
1,Dobermann,Dobermann
2,Micmacs à tire-larigot,Micmacs à tire-larigot
3,La petite moutarde qui monte au nez,La petite moutarde qui monte au nez
4,"Bon Cop, Bad Cop","Bon Cop, Bad Cop"


Toy Story


,version 3,sans durée
0,Toy Story 4,Toy Story 4
1,Toy Story 3,Toy Story 3
2,Toy Story 2,Toy Story 2
3,Monstres & Cie,Monstres & Cie
4,Vice-versa 2,Vice-versa 2


... (6 outputs left out)

Amélie


,version 3,sans durée
0,Paris-Manhattan,Paris-Manhattan
1,L'arnacoeur,L'arnacoeur
2,Simple comme Sylvain,Simple comme Sylvain
3,Lol,20 ans d'écart
4,Amour & Turbulences,Amour & Turbulences


Mission Cléopâtre


,version 3,sans durée
0,Astérix & Obélix contre César,Astérix & Obélix contre César
1,Le Petit Nicolas,Le Petit Nicolas
2,Les vacances du petit Nicolas,Les vacances du petit Nicolas
3,La Guerre des boutons,La Guerre des boutons
4,La nouvelle guerre des boutons,La nouvelle guerre des boutons


> **Constat** : sans la durée, 5 films sur 8 ont exactement les mêmes recommandations (Alien, Taxi, Toy Story, Matrix, Mission Cléopâtre). Pour Intouchables, ce sont les mêmes cinq films, dans un autre ordre. Pour les 2 autres, un seul film change : le 4ᵉ voisin de La Communauté de l'anneau (Le Hobbit : La Désolation de Smaug remplacé par Excalibur) et le 4ᵉ d'Amélie (Lol remplacé par 20 ans d'écart). La durée pèse donc très peu face aux mots-clés, aux genres et à la langue.

> **Décision** : mettre le poids de la durée à 0 dans le moteur final. Les autres poids restent ceux de la version 3 (genres 0,7, année, note et popularité 0,7, mots-clés 1, langue 0,8).

## 6. Les personnes : réalisateurs et acteurs

> **Objectif** : charger le fichier IMDb `title.crew`, qui donne pour chaque film ses réalisateurs (`directors`) et ses scénaristes (`writers`), sous forme d'identifiants `nconst` (nm…). On le lit directement depuis le site d'IMDb, comme dans le notebook de préparation de l'équipe.

In [169]:
crew = pd.read_csv("https://datasets.imdbws.com/title.crew.tsv.gz",
                   sep="\t", na_values="\\N")

In [170]:
print(crew.shape)
crew.head()

(13003096, 3)


,tconst,directors,writers
0,tt0000001,nm0005690,NaN
1,tt0000002,nm0721526,NaN
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,NaN
4,tt0000005,nm0005690,NaN


> **Constat** : `title.crew` compte 13 003 096 lignes, une par titre IMDb : films, mais aussi séries, épisodes, courts métrages… Chaque ligne donne les identifiants des réalisateurs (`directors`) et des scénaristes (`writers`). Une case peut contenir plusieurs identifiants séparés par une virgule, ou être vide (`NaN`).

> **Objectif** : ne garder que les réalisateurs de nos films, dans l'ordre du catalogue. Une jointure à gauche (`how="left"`) part de `films` : on garde ainsi chaque film, une seule fois et à sa place, même s'il est absent de `title.crew`.

In [171]:
realisateurs = films[["tconst"]].merge(crew[["tconst", "directors"]], on="tconst", how="left")
print(realisateurs.shape)
print(realisateurs["directors"].isna().sum())
realisateurs.head()

(6942, 2)
0


,tconst,directors
0,tt0035423,nm0003506
1,tt0064106,nm0001031
2,tt0064177,nm0765121
3,tt0065134,nm0796923
4,tt0065143,nm0000783


> **Constat** : 6 942 lignes, une par film du catalogue, dans le même ordre que `films`. Aucun film n'est sans réalisateur : tous nos films sont dans `title.crew`.

In [172]:
nb_real = realisateurs["directors"].str.split(",").str.len()
nb_real.value_counts()

directors
1     6356
2      489
3       63
4       18
5        4
6        2
7        2
10       1
15       1
8        1
21       1
11       1
36       1
12       1
13       1
Name: count, dtype: int64

In [173]:
# Quelques films à plusieurs réalisateurs
films[nb_real >= 2][["title", "startYear"]].head(10)

,title,startYear
5,Airport,1970
17,Gimme Shelter,1970
31,Tora ! Tora ! Tora !,1970
41,L'Apprentie sorcière,1971
46,L'Inspecteur Harry,1971
56,Le Mans,1971
69,Big Boss,1971
122,Le Petit Monde de Charlotte,1973
141,Robin des Bois,1973
175,Monty Python : Sacré Graal !,1975


In [174]:
plusieurs = films[["title", "startYear"]].copy()
plusieurs["nb_real"] = nb_real
plusieurs.sort_values("nb_real", ascending=False).head(15)

,title,startYear,nb_real
3464,Chacun son cinéma ou Ce petit coup au coeur qu...,2007,36
2814,Paris je t'aime,2006,21
2044,Neon Genesis Evangelion: Death & Rebirth,1997,15
4133,My Movie Project,2013,13
4022,Black Is King,2020,12
3293,"New York, je t'aime",2008,11
478,Métal hurlant,1981,10
2804,Godzilla: Final Wars,2004,8
4242,Dante's Inferno,2010,7
5338,V/H/S 2,2013,7


> **Constat** : 6 356 films ont un seul réalisateur, 489 en ont deux, 63 en ont trois, et quelques-uns beaucoup plus : *Chacun son cinéma* (36), *Paris je t'aime* (21). Ce sont surtout des films à sketches (un réalisateur par segment), des films d'animation (coréalisation fréquente), des duos (frères Coen, Wachowski…), ou des films dont IMDb compte aussi des réalisateurs non crédités. Comme pour les mots-clés, un réalisateur présent dans un seul film du catalogue ne rapprochera aucun film d'un autre : `min_df=2` l'écartera.

> **Objectif** : compter, pour chaque réalisateur, le nombre de films du catalogue qu'il a réalisés (sa fréquence documentaire), avec la même boucle que pour les mots-clés.

In [175]:
compte_real = {}
for texte in realisateurs["directors"]:
    for nm in set(texte.split(",")):
        if nm in compte_real:
            compte_real[nm] += 1
        else:
            compte_real[nm] = 1
print("réalisateurs différents :", len(compte_real))

réalisateurs différents : 3591


In [176]:
nb_uniques_real = 0
for nm in compte_real:
    if compte_real[nm] == 1:
        nb_uniques_real += 1
print("présents dans un seul film :", nb_uniques_real)
print("présents dans au moins 2 films :", len(compte_real) - nb_uniques_real)

présents dans un seul film : 2137
présents dans au moins 2 films : 1454


In [177]:
# Les réalisateurs les plus présents (identifiants seulement, pour l'instant)
pd.Series(compte_real).sort_values(ascending=False).head(10)

nm0000095    44
nm0000229    26
nm0000631    26
nm0000165    23
nm0000142    21
nm0000709    18
nm0001054    18
nm0000500    18
nm0000318    18
nm0001053    17
dtype: int64

> **Constat** : nos 6 942 films ont 3 591 réalisateurs différents. 2 137 d'entre eux (environ 60 %) n'ont réalisé qu'un seul film du catalogue : avec `min_df=2`, il en restera 1 454. Le réalisateur le plus présent signe 44 films du catalogue, les suivants entre 17 et 26.

> **Objectif** : afficher les noms des 10 réalisateurs les plus présents. Les noms sont dans le fichier IMDb `name.basics` (colonnes `nconst` et `primaryName`).

In [178]:
noms = pd.read_csv("https://datasets.imdbws.com/name.basics.tsv.gz",
                   sep="\t", na_values="\\N", usecols=["nconst", "primaryName"])

In [179]:
print(noms.shape)
noms.head()

(15843486, 2)


,nconst,primaryName
0,nm0000001,Fred Astaire
1,nm0000002,Lauren Bacall
2,nm0000003,Brigitte Bardot
3,nm0000004,John Belushi
4,nm0000005,Ingmar Bergman


In [180]:
top_real = pd.Series(compte_real).sort_values(ascending=False).head(10)
top_real = top_real.reset_index()
top_real.columns = ["nconst", "nb_films"]
top_real.merge(noms, on="nconst", how="left")

,nconst,nb_films,primaryName
0,nm0000095,44,Woody Allen
1,nm0000229,26,Steven Spielberg
2,nm0000631,26,Ridley Scott
3,nm0000165,23,Ron Howard
4,nm0000142,21,Clint Eastwood
5,nm0000709,18,Robert Zemeckis
6,nm0001054,18,Joel Coen
7,nm0000500,18,Richard Linklater
8,nm0000318,18,Tim Burton
9,nm0001053,17,Ethan Coen


> **Constat** : les réalisateurs les plus présents sont des auteurs prolifiques : Woody Allen (44 films du catalogue), Steven Spielberg et Ridley Scott (26), Ron Howard (23), Clint Eastwood (21)… Joel Coen (18) et Ethan Coen (17) apparaissent séparément : IMDb compte chaque membre d'un duo comme un réalisateur à part entière.

> **Remarque** : un duo crée deux colonnes qui sont presque toujours à 1 ensemble. Deux films des frères Coen partagent donc deux réalisateurs, là où deux films de Spielberg n'en partagent qu'un : le bloc réalisateurs les rapprochera un peu plus fort. L'effet reste limité, car chaque ligne TF-IDF est ramenée à une longueur de 1.

> **Objectif** : construire le bloc TF-IDF des réalisateurs, comme pour les mots-clés : chaque réalisateur est un « terme », découpé aux virgules (sans espace cette fois), avec `binary=True` et `min_df=2` (un réalisateur présent dans un seul film ne rapproche aucun film d'un autre).

In [181]:
def decouper_virgule(texte):
    return texte.split(",")

tfidf_real = TfidfVectorizer(tokenizer=decouper_virgule, token_pattern=None,
                             binary=True, min_df=2)
mots_real = tfidf_real.fit_transform(realisateurs["directors"])
mots_real.shape

(6942, 1454)

In [182]:
# Films dont le bloc réalisateurs est vide (tous leurs réalisateurs n'ont qu'un film)
longueur_real = np.sqrt(mots_real.multiply(mots_real).sum(axis=1))
print("films sans réalisateur commun :", (longueur_real == 0).sum())

films sans réalisateur commun : 1714


### Modèle version 4 : avec les réalisateurs

> **Objectif** : ajouter le bloc réalisateurs (poids 0,7 pour commencer) au moteur retenu (version 3, durée à 0), puis comparer les recommandations sur les huit films tests.

> **Objectif** : ajouter à `films` une colonne `realisation` avec les noms des réalisateurs. On réduit d'abord `noms` (près de 16 millions de personnes) aux 3 591 réalisateurs du catalogue, puis on en fait un dictionnaire, comme un annuaire : `nom_de["nm0000095"]` donne « Woody Allen ».

In [183]:
noms_real = noms[noms["nconst"].isin(list(compte_real.keys()))]
nom_de = dict(zip(noms_real["nconst"], noms_real["primaryName"]))
nom_de["nm0000095"]

'Woody Allen'

In [184]:
def noms_realisateurs(texte):
    liste = []
    for nm in texte.split(","):
        liste.append(nom_de.get(nm, nm))    # le code si le nom manque
    return ", ".join(liste)

films["realisation"] = realisateurs["directors"].apply(noms_realisateurs)
films[["title", "realisation"]].head()

,title,realisation
0,Kate et Léopold,James Mangold
1,Le boucher,Claude Chabrol
2,Le cerveau d'acier,Joseph Sargent
3,Sierra torride,Don Siegel
4,L'oiseau au plumage de cristal,Dario Argento


> **Objectif** : assembler la matrice `X4` : la matrice du moteur retenu, plus le bloc réalisateurs multiplié par 0,7, puis créer le modèle.

In [185]:
X4 = pd.DataFrame(np.hstack([X3_sans_duree.values,
                             0.7 * mots_real.toarray()]))
modele4 = NearestNeighbors(n_neighbors=6, metric="cosine")
modele4.fit(X4)
X4.shape

(6942, 8463)

> **Objectif** : comparer, pour les huit films tests, les 10 plus proches voisins avec et sans le bloc réalisateurs, en affichant le réalisateur de chaque film. On regarde 10 voisins au lieu de 5 : un film qui recule peut rester dans les 10 premiers, et c'est le nombre de recommandations que l'appli affichera. `n_neighbors=11` remplace, pour cet appel seulement, les 6 voisins fixés à la création du modèle (le premier voisin est le film lui-même).

In [186]:
def recommander4(tconst):
    return recommander(tconst, X4, modele4)

def voisins10(tconst, X, modele):
    i = films[films["tconst"] == tconst].index[0]
    distances, indices = modele.kneighbors(X.iloc[[i]], n_neighbors=11)
    v = films.iloc[indices[0][1:]]
    return (v["title"] + " (" + v["realisation"] + ")").values

def comparer_real(tconst):
    return pd.DataFrame({"version 3": voisins10(tconst, X3_sans_duree, modele3_sans_duree),
                         "avec réalisateurs": voisins10(tconst, X4, modele4)})

In [187]:
for titre in films_tests:
    t = films_tests[titre]
    print(titre, "—", films[films["tconst"] == t]["realisation"].iloc[0])
    display(comparer_real(t))

Alien — Ridley Scott


,version 3,avec réalisateurs
0,Alien 3 (David Fincher),Alien: Covenant (Ridley Scott)
1,Alien: Covenant (Ridley Scott),Prometheus (Ridley Scott)
2,Alien: Romulus (Fede Alvarez),Alien 3 (David Fincher)
3,L'Invasion des profanateurs (Philip Kaufman),Alien: Romulus (Fede Alvarez)
4,"Alien, la résurrection (Jean-Pierre Jeunet)",Talking Dead (Bruce McDonald)
5,La Chose (John Carpenter),Génération Proteus (Donald Cammell)
6,Frissons (David Cronenberg),La Chose (John Carpenter)
7,28 semaines plus tard (Juan Carlos Fresnadillo),Hidden (Jack Sholder)
8,Phantasm (Don Coscarelli),"Alien, la résurrection (Jean-Pierre Jeunet)"
9,Chromosome 3 (David Cronenberg),L'Invasion des profanateurs (Philip Kaufman)


Taxi — Gérard Pirès


,version 3,avec réalisateurs
0,Taxi 2 (Gérard Krawczyk),"Bon Cop, Bad Cop (Erik Canuel)"
1,Dobermann (Jan Kounen),Bon Cop Bad Cop 2 (Alain Desrochers)
2,Micmacs à tire-larigot (Jean-Pierre Jeunet),De père en flic (Émile Gaudreault)
3,La petite moutarde qui monte au nez (Gérard Kr...,Taxi 2 (Gérard Krawczyk)
4,"Bon Cop, Bad Cop (Erik Canuel)",Dobermann (Jan Kounen)
5,Nicky Larson et le parfum de Cupidon (Philippe...,Micmacs à tire-larigot (Jean-Pierre Jeunet)
6,Ripoux contre ripoux (Claude Zidi),La petite moutarde qui monte au nez (Gérard Kr...
7,Bon Cop Bad Cop 2 (Alain Desrochers),Nicky Larson et le parfum de Cupidon (Philippe...
8,De père en flic (Émile Gaudreault),Ripoux contre ripoux (Claude Zidi)
9,30 minutes maximum (Ruben Fleischer),30 minutes maximum (Ruben Fleischer)


Toy Story — John Lasseter


,version 3,avec réalisateurs
0,Toy Story 4 (Josh Cooley),"Cars : Quatre Roues (John Lasseter, Joe Ranft)"
1,Toy Story 3 (Lee Unkrich),"Cars 2 (John Lasseter, Bradford Lewis)"
2,"Toy Story 2 (John Lasseter, Ash Brannon, Lee U...","Toy Story 2 (John Lasseter, Ash Brannon, Lee U..."
3,"Monstres & Cie (Pete Docter, David Silverman, ...","1001 Pattes (John Lasseter, Andrew Stanton)"
4,Vice-versa 2 (Kelsey Mann),Vice-versa 2 (Kelsey Mann)
5,"Vice-versa (Pete Docter, Ronnie Del Carmen)",La Cour de récré: Vive les vacances! (Chuck Sh...
6,"Cars : Quatre Roues (John Lasseter, Joe Ranft)",Toy Story 4 (Josh Cooley)
7,Élémentaire (Peter Sohn),Luca (Enrico Casarosa)
8,La Cour de récré: Vive les vacances! (Chuck Sh...,Cars 3 (Brian Fee)
9,"Le Monde de Nemo (Andrew Stanton, Lee Unkrich)",Toy Story 3 (Lee Unkrich)


... (6 outputs left out)

Amélie — Jean-Pierre Jeunet


,version 3,avec réalisateurs
0,Paris-Manhattan (Sophie Lellouche),Paris-Manhattan (Sophie Lellouche)
1,L'arnacoeur (Pascal Chaumeil),Simple comme Sylvain (Monia Chokri)
2,Simple comme Sylvain (Monia Chokri),Amour & Turbulences (Alexandre Castagnetti)
3,20 ans d'écart (David Moreau),Lol (Lisa Azuelos)
4,Amour & Turbulences (Alexandre Castagnetti),Les émotifs anonymes (Jean-Pierre Améris)
5,Lol (Lisa Azuelos),"Un peu, beaucoup, aveuglément (Clovis Cornillac)"
6,Les émotifs anonymes (Jean-Pierre Améris),Fanfan (Alexandre Jardin)
7,"Un peu, beaucoup, aveuglément (Clovis Cornillac)",Un bonheur n'arrive jamais seul (James Huth)
8,Fanfan (Alexandre Jardin),L'amour dure trois ans (Frédéric Beigbeder)
9,Décalage horaire (Danièle Thompson),Gazon maudit (Josiane Balasko)


Mission Cléopâtre — Alain Chabat


,version 3,avec réalisateurs
0,Astérix & Obélix contre César (Claude Zidi),Astérix & Obélix contre César (Claude Zidi)
1,Le Petit Nicolas (Laurent Tirard),Un cadeau du ciel (Alain Chabat)
2,Les vacances du petit Nicolas (Laurent Tirard),Le Petit Nicolas (Laurent Tirard)
3,La Guerre des boutons (Yann Samuell),La nouvelle guerre des boutons (Christophe Bar...
4,La nouvelle guerre des boutons (Christophe Bar...,Astérix et la Surprise de César (Gaëtan Brizzi...
5,"Ernest et Célestine (Stéphane Aubier, Vincent ...",Les vacances du petit Nicolas (Laurent Tirard)
6,Le Jaguar (Francis Veber),"Astérix et les Vikings (Stefan Fjeldmark, Jesp..."
7,Belle et Sébastien (Nicolas Vanier),"Avril et le monde truqué (Christian Desmares, ..."
8,Astérix et la Surprise de César (Gaëtan Brizzi...,Mune : Le Gardien de la Lune (Alexandre Heboya...
9,"Les 12 Travaux d'Astérix (René Goscinny, Alber...","Une vie de chat (Jean-Loup Felicioli, Alain Ga..."


> **Constat** : avec 10 voisins, le bloc réalisateurs fait remonter des films du même auteur, souvent à bon escient : Prometheus (Ridley Scott) pour Alien, les trois Hobbit et Lovely Bones (Peter Jackson) pour La Communauté de l'anneau, quatre films d'Olivier Nakache et Éric Toledano pour Intouchables, Cars, Cars 2 et 1001 Pattes (John Lasseter) pour Toy Story. Les films de la saga reculent parfois, mais restent dans les 10 premiers : Taxi 2 passe de la 1ʳᵉ à la 4ᵉ place, Toy Story 4 et Toy Story 3 aux 7ᵉ et 10ᵉ places. Des films inattendus apparaissent aussi : Talking Dead, Hidden et Génération Proteus pour Alien, Tron et Equilibrium pour Matrix, Bon Cop Bad Cop 2 et De père en flic, désormais devant Taxi 2, pour Taxi.

> **Objectif** : comprendre d'où viennent ces films inattendus. Hypothèse : ce sont des films dont le bloc réalisateurs est vide (leur réalisateur ne signe aucun autre film du catalogue). Pour Taxi, le bloc de Gérard Pirès est peut-être vide lui aussi. On affiche, pour Taxi et Alien et leurs 10 voisins, si le bloc réalisateurs est vide.

In [188]:
for t in ["tt0152930", "tt0078748"]:
    i = films[films["tconst"] == t].index[0]
    distances, indices = modele4.kneighbors(X4.iloc[[i]], n_neighbors=11)
    for j in indices[0]:
        print(films["title"][j], "| bloc vide :", longueur_real[j, 0] == 0)
    print()

Taxi | bloc vide : True
Bon Cop, Bad Cop | bloc vide : True
Bon Cop Bad Cop 2 | bloc vide : True
De père en flic | bloc vide : True
Taxi 2 | bloc vide : False
Dobermann | bloc vide : False
Micmacs à tire-larigot | bloc vide : False
La petite moutarde qui monte au nez | bloc vide : False
Nicky Larson et le parfum de Cupidon | bloc vide : False
Ripoux contre ripoux | bloc vide : False
30 minutes maximum | bloc vide : False

Alien, le 8ème passager | bloc vide : False
Alien: Covenant | bloc vide : False
Prometheus | bloc vide : False
Alien 3 | bloc vide : False
Alien: Romulus | bloc vide : False
Talking Dead | bloc vide : True
Génération Proteus | bloc vide : True
La Chose | bloc vide : False
Hidden | bloc vide : True
Alien, la résurrection | bloc vide : False
L'Invasion des profanateurs | bloc vide : False



> **Constat** : l'hypothèse est confirmée. Le bloc réalisateurs de Taxi est vide, comme celui des trois films qui passent devant Taxi 2 (Bon Cop, Bad Cop ; Bon Cop Bad Cop 2 ; De père en flic). Pour Alien, les trois films inattendus (Talking Dead, Génération Proteus, Hidden) sont les seuls voisins dont le bloc est vide.

> **Remarque** : la similarité cosinus vaut cos(u, v) = (u · v) / (‖u‖ ‖v‖). Pour un film v qui n'a pas le réalisateur du film de départ, le bloc réalisateurs n'ajoute rien au produit scalaire. Si v a un bloc plein (un autre réalisateur du catalogue), ‖v‖ augmente : son cosinus baisse. Si v a un bloc vide, ‖v‖ ne change pas : son cosinus est intact. Le bloc crée donc trois niveaux : même réalisateur, puis réalisateur absent du reste du catalogue, puis autre réalisateur du catalogue. Les films « orphelins » sont favorisés par rapport aux films d'auteurs connus : c'est un biais de la méthode, d'autant plus fort que le poids du bloc est grand.

> **Objectif** : comparer trois poids pour le bloc réalisateurs (0,7, 0,4 et 0,2), sur les 10 plus proches voisins des huit films tests. Un poids plus faible devrait réduire le biais des films « orphelins », mais aussi l'effet « même réalisateur » : on cherche le meilleur compromis.

In [189]:
def creer_X4(poids):
    X = pd.DataFrame(np.hstack([X3_sans_duree.values,
                                poids * mots_real.toarray()]))
    modele = NearestNeighbors(metric="cosine")
    modele.fit(X)
    return X, modele

X4_04, modele4_04 = creer_X4(0.4)
X4_02, modele4_02 = creer_X4(0.2)

In [190]:
def comparer_poids(tconst):
    return pd.DataFrame({"poids 0,7": voisins10(tconst, X4, modele4),
                         "poids 0,4": voisins10(tconst, X4_04, modele4_04),
                         "poids 0,2": voisins10(tconst, X4_02, modele4_02)})

In [191]:
pd.set_option("display.max_colwidth", 100)
for titre in films_tests:
    t = films_tests[titre]
    print(titre, "—", films[films["tconst"] == t]["realisation"].iloc[0])
    display(comparer_poids(t))

Alien — Ridley Scott


,"poids 0,7","poids 0,4","poids 0,2"
0,Alien: Covenant (Ridley Scott),Alien: Covenant (Ridley Scott),Alien 3 (David Fincher)
1,Prometheus (Ridley Scott),Alien 3 (David Fincher),Alien: Covenant (Ridley Scott)
2,Alien 3 (David Fincher),Alien: Romulus (Fede Alvarez),Alien: Romulus (Fede Alvarez)
3,Alien: Romulus (Fede Alvarez),L'Invasion des profanateurs (Philip Kaufman),L'Invasion des profanateurs (Philip Kaufman)
4,Talking Dead (Bruce McDonald),La Chose (John Carpenter),"Alien, la résurrection (Jean-Pierre Jeunet)"
5,Génération Proteus (Donald Cammell),"Alien, la résurrection (Jean-Pierre Jeunet)",La Chose (John Carpenter)
6,La Chose (John Carpenter),Frissons (David Cronenberg),Frissons (David Cronenberg)
7,Hidden (Jack Sholder),28 semaines plus tard (Juan Carlos Fresnadillo),28 semaines plus tard (Juan Carlos Fresnadillo)
8,"Alien, la résurrection (Jean-Pierre Jeunet)",Talking Dead (Bruce McDonald),Phantasm (Don Coscarelli)
9,L'Invasion des profanateurs (Philip Kaufman),Phantasm (Don Coscarelli),Chromosome 3 (David Cronenberg)


Taxi — Gérard Pirès


,"poids 0,7","poids 0,4","poids 0,2"
0,"Bon Cop, Bad Cop (Erik Canuel)",Taxi 2 (Gérard Krawczyk),Taxi 2 (Gérard Krawczyk)
1,Bon Cop Bad Cop 2 (Alain Desrochers),Dobermann (Jan Kounen),Dobermann (Jan Kounen)
2,De père en flic (Émile Gaudreault),"Bon Cop, Bad Cop (Erik Canuel)",Micmacs à tire-larigot (Jean-Pierre Jeunet)
3,Taxi 2 (Gérard Krawczyk),Bon Cop Bad Cop 2 (Alain Desrochers),La petite moutarde qui monte au nez (Gérard Krawczyk)
4,Dobermann (Jan Kounen),Micmacs à tire-larigot (Jean-Pierre Jeunet),"Bon Cop, Bad Cop (Erik Canuel)"
5,Micmacs à tire-larigot (Jean-Pierre Jeunet),La petite moutarde qui monte au nez (Gérard Krawczyk),Bon Cop Bad Cop 2 (Alain Desrochers)
6,La petite moutarde qui monte au nez (Gérard Krawczyk),De père en flic (Émile Gaudreault),Nicky Larson et le parfum de Cupidon (Philippe Lacheau)
7,Nicky Larson et le parfum de Cupidon (Philippe Lacheau),Nicky Larson et le parfum de Cupidon (Philippe Lacheau),Ripoux contre ripoux (Claude Zidi)
8,Ripoux contre ripoux (Claude Zidi),Ripoux contre ripoux (Claude Zidi),De père en flic (Émile Gaudreault)
9,30 minutes maximum (Ruben Fleischer),30 minutes maximum (Ruben Fleischer),30 minutes maximum (Ruben Fleischer)


Toy Story — John Lasseter


,"poids 0,7","poids 0,4","poids 0,2"
0,"Cars : Quatre Roues (John Lasseter, Joe Ranft)","Toy Story 2 (John Lasseter, Ash Brannon, Lee Unkrich)",Toy Story 4 (Josh Cooley)
1,"Cars 2 (John Lasseter, Bradford Lewis)","Cars : Quatre Roues (John Lasseter, Joe Ranft)","Toy Story 2 (John Lasseter, Ash Brannon, Lee Unkrich)"
2,"Toy Story 2 (John Lasseter, Ash Brannon, Lee Unkrich)",Toy Story 4 (Josh Cooley),Toy Story 3 (Lee Unkrich)
3,"1001 Pattes (John Lasseter, Andrew Stanton)",Toy Story 3 (Lee Unkrich),"Cars : Quatre Roues (John Lasseter, Joe Ranft)"
4,Vice-versa 2 (Kelsey Mann),"Cars 2 (John Lasseter, Bradford Lewis)",Vice-versa 2 (Kelsey Mann)
5,La Cour de récré: Vive les vacances! (Chuck Sheetz),"1001 Pattes (John Lasseter, Andrew Stanton)","Monstres & Cie (Pete Docter, David Silverman, Lee Unkrich)"
6,Toy Story 4 (Josh Cooley),Vice-versa 2 (Kelsey Mann),"Vice-versa (Pete Docter, Ronnie Del Carmen)"
7,Luca (Enrico Casarosa),La Cour de récré: Vive les vacances! (Chuck Sheetz),La Cour de récré: Vive les vacances! (Chuck Sheetz)
8,Cars 3 (Brian Fee),"Monstres & Cie (Pete Docter, David Silverman, Lee Unkrich)",Élémentaire (Peter Sohn)
9,Toy Story 3 (Lee Unkrich),Luca (Enrico Casarosa),"1001 Pattes (John Lasseter, Andrew Stanton)"


... (6 outputs left out)

Amélie — Jean-Pierre Jeunet


,"poids 0,7","poids 0,4","poids 0,2"
0,Paris-Manhattan (Sophie Lellouche),Paris-Manhattan (Sophie Lellouche),Paris-Manhattan (Sophie Lellouche)
1,Simple comme Sylvain (Monia Chokri),Simple comme Sylvain (Monia Chokri),L'arnacoeur (Pascal Chaumeil)
2,Amour & Turbulences (Alexandre Castagnetti),Amour & Turbulences (Alexandre Castagnetti),Simple comme Sylvain (Monia Chokri)
3,Lol (Lisa Azuelos),Lol (Lisa Azuelos),Amour & Turbulences (Alexandre Castagnetti)
4,Les émotifs anonymes (Jean-Pierre Améris),Les émotifs anonymes (Jean-Pierre Améris),Lol (Lisa Azuelos)
5,"Un peu, beaucoup, aveuglément (Clovis Cornillac)",L'arnacoeur (Pascal Chaumeil),Les émotifs anonymes (Jean-Pierre Améris)
6,Fanfan (Alexandre Jardin),"Un peu, beaucoup, aveuglément (Clovis Cornillac)",20 ans d'écart (David Moreau)
7,Un bonheur n'arrive jamais seul (James Huth),Fanfan (Alexandre Jardin),"Un peu, beaucoup, aveuglément (Clovis Cornillac)"
8,L'amour dure trois ans (Frédéric Beigbeder),Un bonheur n'arrive jamais seul (James Huth),Fanfan (Alexandre Jardin)
9,Gazon maudit (Josiane Balasko),L'amour dure trois ans (Frédéric Beigbeder),Un bonheur n'arrive jamais seul (James Huth)


Mission Cléopâtre — Alain Chabat


,"poids 0,7","poids 0,4","poids 0,2"
0,Astérix & Obélix contre César (Claude Zidi),Astérix & Obélix contre César (Claude Zidi),Astérix & Obélix contre César (Claude Zidi)
1,Un cadeau du ciel (Alain Chabat),Le Petit Nicolas (Laurent Tirard),Le Petit Nicolas (Laurent Tirard)
2,Le Petit Nicolas (Laurent Tirard),Les vacances du petit Nicolas (Laurent Tirard),Les vacances du petit Nicolas (Laurent Tirard)
3,La nouvelle guerre des boutons (Christophe Barratier),La nouvelle guerre des boutons (Christophe Barratier),La nouvelle guerre des boutons (Christophe Barratier)
4,"Astérix et la Surprise de César (Gaëtan Brizzi, Paul Brizzi)",Un cadeau du ciel (Alain Chabat),La Guerre des boutons (Yann Samuell)
5,Les vacances du petit Nicolas (Laurent Tirard),"Astérix et la Surprise de César (Gaëtan Brizzi, Paul Brizzi)","Ernest et Célestine (Stéphane Aubier, Vincent Patar, Benjamin Renner)"
6,"Astérix et les Vikings (Stefan Fjeldmark, Jesper Møller)","Ernest et Célestine (Stéphane Aubier, Vincent Patar, Benjamin Renner)",Le Jaguar (Francis Veber)
7,"Avril et le monde truqué (Christian Desmares, Franck Ekinci)",La Guerre des boutons (Yann Samuell),"Astérix et la Surprise de César (Gaëtan Brizzi, Paul Brizzi)"
8,"Mune : Le Gardien de la Lune (Alexandre Heboyan, Benoît Philippon)","Astérix et les Vikings (Stefan Fjeldmark, Jesper Møller)",Belle et Sébastien (Nicolas Vanier)
9,"Une vie de chat (Jean-Loup Felicioli, Alain Gagnol)",Le Jaguar (Francis Veber),"Les 12 Travaux d'Astérix (René Goscinny, Albert Uderzo)"


> **Constat** : à 0,7, le bloc réalisateurs domine : Taxi 2 recule au 4ᵉ rang, Toy Story 3 au 10ᵉ, et les films au bloc vide (Talking Dead, Bon Cop, Bad Cop…) remontent. À 0,2, on retrouve presque la version 3 : l'effet « même réalisateur » disparaît (un seul film de Toledano et Nakache pour Intouchables, plus de film d'Alain Chabat pour Mission Cléopâtre). À 0,4, le compromis est le meilleur : Taxi 2 revient en tête, les trois suites de Toy Story sont dans les quatre premiers avec Cars et 1001 Pattes, les trois Hobbit restent pour La Communauté de l'anneau, deux films de Toledano et Nakache pour Intouchables ; Talking Dead et Tron reculent au 9ᵉ rang. Seule perte : Prometheus sort des dix voisins d'Alien. Amélie ne change à aucun poids.

> **Décision** : retenir un poids de 0,4 pour le bloc réalisateurs. C'est un compromis : le réglage idéal dépend de ce que l'on attend (des films du même auteur, ou des films qui se ressemblent par leur contenu). C'est précisément ce que la table de mixage de l'appli permettra d'explorer.

### Les acteurs

> **Objectif** : charger le fichier IMDb `title.principals`, qui donne pour chaque titre ses principales personnes (acteurs, réalisateur, compositeur…), avec leur rang au générique (`ordering`), leur métier (`category`) et, pour les acteurs, le personnage joué (`characters`). Le fichier compte près de 100 millions de lignes : on le lit par morceaux de 2 millions de lignes (`chunksize`), comme le fichier `akas` dans le notebook des titres français, et on ne garde de chaque morceau que nos films.

In [192]:
colonnes = ["tconst", "ordering", "nconst", "category", "characters"]
nos_films = set(films["tconst"])

morceaux = pd.read_csv("https://datasets.imdbws.com/title.principals.tsv.gz",
                       sep="\t", na_values="\\N", usecols=colonnes,
                       chunksize=2_000_000)

gardes = []
for morceau in morceaux:
    gardes.append(morceau[morceau["tconst"].isin(nos_films)])

principals = pd.concat(gardes, ignore_index=True)

In [193]:
print(principals.shape)
principals.head(15)

(149987, 5)


,tconst,ordering,nconst,category,characters
0,tt0035423,1,nm0000212,actress,"[""Kate McKay""]"
1,tt0035423,2,nm0413168,actor,"[""Leopold""]"
2,tt0035423,3,nm0000630,actor,"[""Stuart Besser""]"
3,tt0035423,4,nm0005227,actor,"[""Charlie McKay""]"
4,tt0035423,5,nm0005169,actress,"[""Darci""]"
5,tt0035423,6,nm0925966,actor,"[""J.J. Camden""]"
6,tt0035423,7,nm0925717,actor,"[""Uncle Millard""]"
7,tt0035423,8,nm0336960,actor,"[""Dr. Geisler""]"
8,tt0035423,9,nm0821817,actor,"[""Colleague Bob""]"
9,tt0035423,10,nm0697069,actor,"[""Ad Executive Phil""]"


> **Constat** : pour nos 6 942 films, `title.principals` garde 149 987 lignes, soit une vingtaine de personnes par film. Chaque ligne donne une personne (`nconst`), son rang au générique (`ordering`), son métier (`category` : actrice, acteur, réalisateur, scénariste, producteur, compositeur…) et, pour les acteurs, le personnage joué (`characters`, sous forme de liste, par exemple `["Kate McKay"]`). Dans l'exemple de Kate et Léopold, les acteurs occupent les premiers rangs (1 à 10), avant l'équipe technique.

> **Objectif** : voir les différents métiers présents, pour ne garder ensuite que les acteurs et actrices.

In [194]:
principals["category"].value_counts()

category
actor                  47712
actress                23166
producer               16251
writer                 15475
editor                  9061
casting_director        8829
composer                7798
director                7519
cinematographer         7057
production_designer     6597
self                     397
archive_footage          122
archive_sound              3
Name: count, dtype: int64

> **Objectif** : ne garder que les acteurs et actrices, puis regarder combien chaque film en compte.

In [195]:
filtre_acteurs = principals["category"].isin(["actor", "actress"])
acteurs = principals[filtre_acteurs]
print(acteurs.shape)
acteurs.groupby("tconst").size().describe()

(70878, 5)


count    6904.000000
mean       10.266222
std         1.353106
min         1.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        30.000000
dtype: float64

> **Constat** : parmi les 149 987 lignes, 70 878 concernent des acteurs (47 712) ou des actrices (23 166). Le reste, c'est l'équipe : production, scénario, montage, musique, image… On trouve aussi 397 lignes `self` : des personnes qui jouent leur propre rôle, dans des documentaires ou des films de concert. 6 904 films ont au moins un acteur, et la plupart en ont exactement 10 (la médiane et les deux quartiles valent 10) : IMDb limite cette liste aux principaux rôles.

> **Objectif** : compter et afficher les films qui n'ont ni acteur ni actrice dans `title.principals`.

In [196]:
sans_acteur = ~films["tconst"].isin(acteurs["tconst"])
print(sans_acteur.sum())
films[sans_acteur][["title", "startYear", "genres"]].head(15)

38


,title,startYear,genres
17,Gimme Shelter,1970,"Documentary,Music,Thriller"
204,Grey Gardens,1975,"Comedy,Documentary,Drama"
722,Shoah,1985,"Documentary,History,War"
1379,Crumb,1994,"Biography,Comedy,Documentary"
2496,Jackass: Le film,2002,"Action,Comedy,Documentary"
2533,Spellbound,2002,"Comedy,Documentary,Thriller"
2626,Fahrenheit 9/11,2004,"Documentary,Drama,War"
2967,Le Nerf de la guerre,2005,"Documentary,History,War"
3202,Jackass Deux: Le film,2006,"Action,Comedy,Documentary"
3302,Religolo,2008,"Comedy,Documentary,War"


> **Constat** : 38 films n'ont ni acteur ni actrice. Ce sont presque tous des documentaires (Shoah, Gimme Shelter, Fahrenheit 9/11…) ou des films Jackass : les personnes qu'on y voit jouent leur propre rôle et sont classées `self`, pas `actor`. Seule exception visible : Superman II: The Richard Donner Cut, un nouveau montage réalisé à partir d'images tournées en 1978-1980.

> **Objectif** : garder aussi la catégorie `self` (personnes qui jouent leur propre rôle), pour que des films comme les Jackass, qui partagent les mêmes personnes, puissent se rapprocher. On compte ensuite les films qui restent sans personne.

In [197]:
filtre_acteurs = principals["category"].isin(["actor", "actress", "self"])
acteurs = principals[filtre_acteurs]
sans_acteur = ~films["tconst"].isin(acteurs["tconst"])
print(acteurs.shape)
print(sans_acteur.sum())
films[sans_acteur][["title", "startYear", "genres"]]

(71275, 5)
4


,title,startYear,genres
3341,Superman II: The Richard Donner Cut,2006,"Action,Adventure,Romance"
6180,Flow,2024,"Adventure,Animation,Family"
6293,Twin Peaks : Les Pièces manquantes du dossier,2014,"Drama,Horror,Mystery"
6415,Kill Bill: The Whole Bloody Affair,2004,"Action,Crime,Thriller"


> **Constat** : en ajoutant `self`, on passe de 70 878 à 71 275 lignes (les 397 lignes `self`), et il ne reste que 4 films sans personne : trois remontages de films existants (Superman II: The Richard Donner Cut, Twin Peaks : Les Pièces manquantes du dossier, Kill Bill: The Whole Bloody Affair) et Flow, un film d'animation sans dialogue dont les héros sont des animaux. Pour ces 4 films, le bloc acteurs sera vide.

> **Objectif** : garder, pour chaque film, ses 4 premiers acteurs dans l'ordre du générique (`ordering`) : ce sont les têtes d'affiche. Les seconds rôles rapprocheraient des films sur des apparitions secondaires. On regroupe ensuite leurs identifiants dans une seule chaîne par film, séparés par une virgule, comme la colonne `directors`.

In [198]:
acteurs = acteurs.sort_values(["tconst", "ordering"])
tetes = acteurs.groupby("tconst").head(4)
print(tetes.shape)
tetes.head(8)

(27728, 5)


,tconst,ordering,nconst,category,characters
0,tt0035423,1,nm0000212,actress,"[""Kate McKay""]"
1,tt0035423,2,nm0413168,actor,"[""Leopold""]"
2,tt0035423,3,nm0000630,actor,"[""Stuart Besser""]"
3,tt0035423,4,nm0005227,actor,"[""Charlie McKay""]"
21,tt0064106,1,nm0000804,actress,"[""Hélène""]"
22,tt0064106,2,nm0946179,actor,"[""Popaul""]"
23,tt0064106,3,nm0946179,actor,"[""Paul Thomas""]"
24,tt0064106,4,nm0664207,actor,"[""Angelo""]"


> **Constat** : il reste 27 728 lignes, soit au plus 4 personnes par film, dans l'ordre du générique. Un même acteur peut apparaître deux fois dans un film quand il y tient deux rôles : dans Le boucher, l'identifiant nm0946179 est aux rangs 2 et 3 (« Popaul » et « Paul Thomas », deux noms du même personnage). Ce doublon ne pose pas de problème : `binary=True` le comptera une seule fois, comme les mots-clés répétés.

> **Objectif** : regrouper les identifiants des têtes d'affiche de chaque film dans une seule chaîne, séparés par une virgule, puis la placer dans l'ordre de `films` par une jointure à gauche. Les 4 films sans personne reçoivent une chaîne vide.

In [199]:
liste_acteurs = tetes.groupby("tconst")["nconst"].agg(",".join).reset_index()
acteurs_films = films[["tconst"]].merge(liste_acteurs, on="tconst", how="left")
acteurs_films["nconst"] = acteurs_films["nconst"].fillna("")
print(acteurs_films.shape)
acteurs_films.head()

(6942, 2)


,tconst,nconst
0,tt0035423,"nm0000212,nm0413168,nm0000630,nm0005227"
1,tt0064106,"nm0000804,nm0946179,nm0946179,nm0664207"
2,tt0064177,"nm0000967,nm0164540,nm0684521,nm0769974"
3,tt0065134,"nm0000142,nm0000511,nm0299578,nm0605494"
4,tt0065143,"nm0615540,nm0447648,nm0758066,nm0719960"


> **Constat** : 6 942 lignes, une par film, dans l'ordre de `films`. Chaque case contient les identifiants des têtes d'affiche, séparés par une virgule ; un acteur qui tient deux rôles apparaît deux fois (Le boucher, ligne 1).

> **Objectif** : compter, pour chaque tête d'affiche, le nombre de films du catalogue où elle apparaît, avec la même boucle que pour les réalisateurs. Le test `if nm != ""` ignore la chaîne vide des 4 films sans personne.

In [200]:
compte_act = {}
for texte in acteurs_films["nconst"]:
    for nm in set(texte.split(",")):
        if nm != "":
            if nm in compte_act:
                compte_act[nm] += 1
            else:
                compte_act[nm] = 1

nb_uniques_act = 0
for nm in compte_act:
    if compte_act[nm] == 1:
        nb_uniques_act += 1
print("têtes d'affiche différentes :", len(compte_act))
print("présentes dans un seul film :", nb_uniques_act)
print("présentes dans au moins 2 films :", len(compte_act) - nb_uniques_act)

têtes d'affiche différentes : 11657
présentes dans un seul film : 7884
présentes dans au moins 2 films : 3773


In [201]:
top_act = pd.Series(compte_act).sort_values(ascending=False).head(10)
top_act = top_act.reset_index()
top_act.columns = ["nconst", "nb_films"]
top_act.merge(noms, on="nconst", how="left")

,nconst,nb_films,primaryName
0,nm0000158,46,Tom Hanks
1,nm0000134,45,Robert De Niro
2,nm0000115,44,Nicolas Cage
3,nm0000168,40,Samuel L. Jackson
4,nm0005351,39,Ryan Reynolds
5,nm0000246,39,Bruce Willis
6,nm0000329,39,Jackie Chan
7,nm0000242,38,Mark Wahlberg
8,nm0000093,36,Brad Pitt
9,nm0000136,35,Johnny Depp


> **Constat** : nos films comptent 11 657 têtes d'affiche différentes. 7 884 d'entre elles (environ 68 %) n'apparaissent que dans un seul film du catalogue : avec `min_df=2`, il en restera 3 773. Les plus présentes sont des stars très prolifiques : Tom Hanks (46 films), Robert De Niro (45), Nicolas Cage (44), Samuel L. Jackson (40)… Comme « alien » parmi les mots-clés, une star présente dans des dizaines de films aura un poids IDF faible : partager Tom Hanks rapprochera moins deux films que partager un acteur plus rare.

> **Objectif** : construire le bloc TF-IDF des acteurs, comme celui des réalisateurs (`binary=True`, `min_df=2`). La fonction de découpage ignore la chaîne vide des 4 films sans personne : sinon, ce « terme » vide, présent dans 4 films, serait gardé par `min_df=2` et rapprocherait ces films sans raison.

In [202]:
def decouper_acteurs(texte):
    liste = []
    for nm in texte.split(","):
        if nm != "":
            liste.append(nm)
    return liste

tfidf_act = TfidfVectorizer(tokenizer=decouper_acteurs, token_pattern=None,
                            binary=True, min_df=2)
mots_act = tfidf_act.fit_transform(acteurs_films["nconst"])
print(mots_act.shape)

longueur_act = np.sqrt(mots_act.multiply(mots_act).sum(axis=1))
print("films sans acteur commun :", (longueur_act == 0).sum())

(6942, 3773)
films sans acteur commun : 434


> **Constat** : le bloc acteurs a 3 773 colonnes, une par tête d'affiche présente dans au moins deux films. Seuls 434 films (environ 6 % du catalogue) ont un bloc vide, contre 1 714 pour les réalisateurs : les acteurs tournent beaucoup plus de films que les réalisateurs n'en réalisent. Le biais des films « orphelins » devrait donc être bien plus faible.

In [203]:
tetes["category"].value_counts()

category
actor      18030
actress     9535
self         163
Name: count, dtype: int64

### Modèle version 5 : avec les acteurs

> **Objectif** : pour lire les comparaisons, afficher à côté de chaque film le nom de ses deux premières têtes d'affiche (le moteur, lui, en utilise quatre). Pour cela, on ajoute les acteurs à l'annuaire `nom_de`, qui ne contient pour l'instant que les réalisateurs.

In [204]:
noms_act = noms[noms["nconst"].isin(list(compte_act.keys()))]
for nm, nom in zip(noms_act["nconst"], noms_act["primaryName"]):
    nom_de[nm] = nom
nom_de["nm0000158"]

'Tom Hanks'

In [205]:
def deux_acteurs(texte):
    liste = []
    for nm in decouper_acteurs(texte)[:2]:
        liste.append(nom_de.get(nm, nm))
    return ", ".join(liste)

films["casting"] = acteurs_films["nconst"].apply(deux_acteurs)
films[["title", "realisation", "casting"]].head()

,title,realisation,casting
0,Kate et Léopold,James Mangold,"Meg Ryan, Hugh Jackman"
1,Le boucher,Claude Chabrol,"Stéphane Audran, Jean Yanne"
2,Le cerveau d'acier,Joseph Sargent,"Eric Braeden, Susan Clark"
3,Sierra torride,Don Siegel,"Clint Eastwood, Shirley MacLaine"
4,L'oiseau au plumage de cristal,Dario Argento,"Tony Musante, Suzy Kendall"


> **Objectif** : vérifier que les personnes `self` apparaissent bien dans `casting`, sur les films Jackass, où les participants jouent leur propre rôle.

In [206]:
films[films["title"].str.contains("Jackass")][["title", "casting"]]

,title,casting
2496,Jackass: Le film,"Johnny Knoxville, Bam Margera"
3202,Jackass Deux: Le film,"Johnny Knoxville, Steve-O"
3730,Jackass 3D,"Johnny Knoxville, Steve-O"
3797,Jackass Forever,"Johnny Knoxville, Steve-O"
6015,Jackass: le meilleur pour la fin,"Johnny Knoxville, Steve-O"


> **Constat** : les 5 films Jackass ont bien des têtes d'affiche, alors que leurs participants sont classés `self` : Johnny Knoxville dans les 5, Steve-O dans 4. Ces films pourront donc se rapprocher entre eux par le bloc acteurs.

> **Objectif** : assembler la matrice `X5` : la version 4 (moteur retenu et réalisateurs au poids 0,4), plus le bloc acteurs au poids 0,4 pour commencer, puis créer le modèle.

In [207]:
X5 = pd.DataFrame(np.hstack([X4_04.values, 0.4 * mots_act.toarray()]))
modele5 = NearestNeighbors(metric="cosine")
modele5.fit(X5)
X5.shape

(6942, 12236)

> **Constat** : `X5` a 6 942 lignes et 12 236 colonnes : les 8 463 colonnes de la version 4, plus les 3 773 du bloc acteurs.

> **Objectif** : comparer, pour les huit films tests, les 10 plus proches voisins de la version 4 (sans acteurs) et de la version 5 (avec acteurs). En tête de chaque tableau, les quatre têtes d'affiche du film de départ ; pour chaque voisin, on affiche entre crochets les têtes d'affiche qu'il partage avec le film de départ (parmi les quatre utilisées par le moteur) : c'est exactement ce que le bloc acteurs prend en compte.

In [208]:
def tetes_affiche(tconst):
    texte = acteurs_films[acteurs_films["tconst"] == tconst]["nconst"].iloc[0]
    liste = []
    for nm in decouper_acteurs(texte):
        liste.append(nom_de.get(nm, nm))
    return ", ".join(liste)

def acteurs_communs(t1, t2):
    a1 = set(decouper_acteurs(acteurs_films[acteurs_films["tconst"] == t1]["nconst"].iloc[0]))
    a2 = set(decouper_acteurs(acteurs_films[acteurs_films["tconst"] == t2]["nconst"].iloc[0]))
    liste = []
    for nm in a1 & a2:
        liste.append(nom_de.get(nm, nm))
    return ", ".join(liste)

def voisins10_communs(tconst, X, modele):
    i = films[films["tconst"] == tconst].index[0]
    distances, indices = modele.kneighbors(X.iloc[[i]], n_neighbors=11)
    lignes = []
    for j in indices[0][1:]:
        communs = acteurs_communs(tconst, films["tconst"][j])
        lignes.append(films["title"][j] + " [" + communs + "]")
    return lignes

def comparer_act(tconst):
    return pd.DataFrame({"version 4": voisins10_communs(tconst, X4_04, modele4_04),
                         "avec acteurs": voisins10_communs(tconst, X5, modele5)})

In [209]:
for titre in films_tests:
    t = films_tests[titre]
    print(titre, "—", tetes_affiche(t))
    display(comparer_act(t))

Alien — Sigourney Weaver, Tom Skerritt, John Hurt, Veronica Cartwright


,version 4,avec acteurs
0,Alien: Covenant [],Alien: Covenant []
1,Alien 3 [Sigourney Weaver],Alien 3 [Sigourney Weaver]
2,Alien: Romulus [],L'Invasion des profanateurs [Veronica Cartwright]
3,L'Invasion des profanateurs [Veronica Cartwright],Alien: Romulus []
4,La Chose [],"Alien, la résurrection [Sigourney Weaver]"
5,"Alien, la résurrection [Sigourney Weaver]",La Chose []
6,Frissons [],Frissons []
7,28 semaines plus tard [],28 semaines plus tard []
8,Talking Dead [],Talking Dead []
9,Phantasm [],Phantasm []


Taxi — Samy Naceri, Frédéric Diefenthal, Marion Cotillard, Manuela Gourary


,version 4,avec acteurs
0,"Taxi 2 [Frédéric Diefenthal, Samy Naceri, Marion Cotillard]","Taxi 2 [Frédéric Diefenthal, Samy Naceri, Marion Cotillard]"
1,Dobermann [],"Bon Cop, Bad Cop []"
2,"Bon Cop, Bad Cop []",Dobermann []
3,Bon Cop Bad Cop 2 [],Micmacs à tire-larigot []
4,Micmacs à tire-larigot [],Bon Cop Bad Cop 2 []
5,La petite moutarde qui monte au nez [],La petite moutarde qui monte au nez []
6,De père en flic [],De père en flic []
7,Nicky Larson et le parfum de Cupidon [],Nicky Larson et le parfum de Cupidon []
8,Ripoux contre ripoux [],Ripoux contre ripoux []
9,30 minutes maximum [],30 minutes maximum []


Toy Story — Tom Hanks, Tim Allen, Don Rickles, Jim Varney


,version 4,avec acteurs
0,"Toy Story 2 [Tom Hanks, Tim Allen]","Toy Story 2 [Tom Hanks, Tim Allen]"
1,Cars : Quatre Roues [],"Toy Story 4 [Tom Hanks, Tim Allen]"
2,"Toy Story 4 [Tom Hanks, Tim Allen]",Cars : Quatre Roues []
3,"Toy Story 3 [Tom Hanks, Tim Allen]","Toy Story 3 [Tom Hanks, Tim Allen]"
4,Cars 2 [],La Cour de récré: Vive les vacances! []
5,1001 Pattes [],Cars 2 []
6,Vice-versa 2 [],1001 Pattes []
7,La Cour de récré: Vive les vacances! [],Vice-versa 2 []
8,Monstres & Cie [],Monstres & Cie []
9,Luca [],Vice-versa []


... (6 outputs left out)

Amélie — Audrey Tautou, Mathieu Kassovitz, Rufus, Lorella Cravotta


,version 4,avec acteurs
0,Paris-Manhattan [],Paris-Manhattan []
1,Simple comme Sylvain [],Les émotifs anonymes [Lorella Cravotta]
2,Amour & Turbulences [],Simple comme Sylvain []
3,Lol [],Amour & Turbulences []
4,Les émotifs anonymes [Lorella Cravotta],Lol []
5,L'arnacoeur [],L'arnacoeur []
6,"Un peu, beaucoup, aveuglément []","Un peu, beaucoup, aveuglément []"
7,Fanfan [],Fanfan []
8,Un bonheur n'arrive jamais seul [],Un bonheur n'arrive jamais seul []
9,L'amour dure trois ans [],L'amour dure trois ans []


Mission Cléopâtre — Gérard Depardieu, Christian Clavier, Jamel Debbouze, Monica Bellucci


,version 4,avec acteurs
0,"Astérix & Obélix contre César [Christian Clavier, Gérard Depardieu]","Astérix & Obélix contre César [Christian Clavier, Gérard Depardieu]"
1,Le Petit Nicolas [],Le Petit Nicolas []
2,Les vacances du petit Nicolas [],Les vacances du petit Nicolas []
3,La nouvelle guerre des boutons [],La nouvelle guerre des boutons []
4,Un cadeau du ciel [],Un cadeau du ciel []
5,Astérix et la Surprise de César [],Astérix et la Surprise de César []
6,Ernest et Célestine [],Ernest et Célestine []
7,La Guerre des boutons [],Astérix et les Vikings []
8,Astérix et les Vikings [],La Guerre des boutons []
9,Le Jaguar [],Belle et Sébastien []


> **Constat** : au poids 0,4, le bloc acteurs change peu les recommandations. Il ne fait apparaître aucun nouveau film de saga (ils étaient déjà là grâce aux mots-clés et aux réalisateurs), mais il fait remonter un peu les films qui partagent des têtes d'affiche : Toy Story 4 passe du 3ᵉ au 2ᵉ rang (Tim Allen, Tom Hanks), Alien, la résurrection du 6ᵉ au 5ᵉ (Sigourney Weaver), Les émotifs anonymes du 5ᵉ au 2ᵉ pour Amélie (Lorella Cravotta). Une seule arrivée s'explique par les acteurs : Les Petits Mouchoirs (François Cluzet) pour Intouchables. Vice-versa (Toy Story) et Belle et Sébastien (Mission Cléopâtre) arrivent aussi, sans acteur en commun.

> **Remarque** : la plupart des voisins ne partagent aucune tête d'affiche avec le film de départ. C'est normal : deux films ne partagent un de leurs quatre premiers acteurs que s'ils ont un lien fort (une saga, un acteur fétiche). Le bloc acteurs agit donc comme un bonus pour ces rares films, et non comme les genres, que presque tous les voisins partagent.

> **Objectif** : comparer trois poids pour le bloc acteurs (0,4, 0,7 et 1), sur les 10 plus proches voisins des huit films tests, en gardant les réalisateurs au poids 0,4. On cherche à partir de quel poids le bloc acteurs devient visible, et s'il fait apparaître des films choisis pour leurs acteurs plutôt que pour leur contenu. Entre crochets : les têtes d'affiche partagées avec le film de départ.

In [210]:
def creer_X5(poids):
    X = pd.DataFrame(np.hstack([X4_04.values, poids * mots_act.toarray()]))
    modele = NearestNeighbors(metric="cosine")
    modele.fit(X)
    return X, modele

X5_07, modele5_07 = creer_X5(0.7)
X5_10, modele5_10 = creer_X5(1)

In [211]:
def comparer_poids_act(tconst):
    return pd.DataFrame({"poids 0,4": voisins10_communs(tconst, X5, modele5),
                         "poids 0,7": voisins10_communs(tconst, X5_07, modele5_07),
                         "poids 1": voisins10_communs(tconst, X5_10, modele5_10)})

In [212]:
for titre in films_tests:
    t = films_tests[titre]
    print(titre, "—", tetes_affiche(t))
    display(comparer_poids_act(t))

Alien — Sigourney Weaver, Tom Skerritt, John Hurt, Veronica Cartwright


,"poids 0,4","poids 0,7",poids 1
0,Alien: Covenant [],Alien: Covenant [],Alien 3 [Sigourney Weaver]
1,Alien 3 [Sigourney Weaver],Alien 3 [Sigourney Weaver],L'Invasion des profanateurs [Veronica Cartwright]
2,L'Invasion des profanateurs [Veronica Cartwright],L'Invasion des profanateurs [Veronica Cartwright],Alien: Covenant []
3,Alien: Romulus [],"Alien, la résurrection [Sigourney Weaver]","Alien, la résurrection [Sigourney Weaver]"
4,"Alien, la résurrection [Sigourney Weaver]",Alien: Romulus [],Cube []
5,La Chose [],La Chose [],Les Clowns tueurs venus d'ailleurs []
6,Frissons [],Cube [],Elmer le remue-méninges []
7,28 semaines plus tard [],28 semaines plus tard [],Alien: Romulus []
8,Talking Dead [],Frissons [],Psycho Goreman []
9,Phantasm [],Les Clowns tueurs venus d'ailleurs [],Bad Taste []


Taxi — Samy Naceri, Frédéric Diefenthal, Marion Cotillard, Manuela Gourary


,"poids 0,4","poids 0,7",poids 1
0,"Taxi 2 [Frédéric Diefenthal, Samy Naceri, Marion Cotillard]","Taxi 2 [Frédéric Diefenthal, Samy Naceri, Marion Cotillard]","Taxi 2 [Frédéric Diefenthal, Samy Naceri, Marion Cotillard]"
1,"Bon Cop, Bad Cop []","Bon Cop, Bad Cop []","Bon Cop, Bad Cop []"
2,Dobermann [],Dobermann [],Dobermann []
3,Micmacs à tire-larigot [],Micmacs à tire-larigot [],Micmacs à tire-larigot []
4,Bon Cop Bad Cop 2 [],Bon Cop Bad Cop 2 [],Bon Cop Bad Cop 2 []
5,La petite moutarde qui monte au nez [],La petite moutarde qui monte au nez [],La petite moutarde qui monte au nez []
6,De père en flic [],De père en flic [],De père en flic []
7,Nicky Larson et le parfum de Cupidon [],Nicky Larson et le parfum de Cupidon [],Nicky Larson et le parfum de Cupidon []
8,Ripoux contre ripoux [],Ripoux contre ripoux [],30 minutes maximum []
9,30 minutes maximum [],30 minutes maximum [],Ripoux contre ripoux []


Toy Story — Tom Hanks, Tim Allen, Don Rickles, Jim Varney


,"poids 0,4","poids 0,7",poids 1
0,"Toy Story 2 [Tom Hanks, Tim Allen]","Toy Story 2 [Tom Hanks, Tim Allen]","Toy Story 2 [Tom Hanks, Tim Allen]"
1,"Toy Story 4 [Tom Hanks, Tim Allen]","Toy Story 4 [Tom Hanks, Tim Allen]","Toy Story 4 [Tom Hanks, Tim Allen]"
2,Cars : Quatre Roues [],"Toy Story 3 [Tom Hanks, Tim Allen]","Toy Story 3 [Tom Hanks, Tim Allen]"
3,"Toy Story 3 [Tom Hanks, Tim Allen]",La Cour de récré: Vive les vacances! [],La Cour de récré: Vive les vacances! []
4,La Cour de récré: Vive les vacances! [],Cars : Quatre Roues [],Space Jam []
5,Cars 2 [],Space Jam [],Les Simpson: Le film []
6,1001 Pattes [],Cars 2 [],Voyage vers la Lune []
7,Vice-versa 2 [],Vice-versa 2 [],Cars : Quatre Roues []
8,Monstres & Cie [],1001 Pattes [],Snoopy et les Peanuts: Le film []
9,Vice-versa [],Les Simpson: Le film [],Abominable []


... (6 outputs left out)

Amélie — Audrey Tautou, Mathieu Kassovitz, Rufus, Lorella Cravotta


,"poids 0,4","poids 0,7",poids 1
0,Paris-Manhattan [],Paris-Manhattan [],Les émotifs anonymes [Lorella Cravotta]
1,Les émotifs anonymes [Lorella Cravotta],Les émotifs anonymes [Lorella Cravotta],Le battement d'ailes du papillon [Audrey Tautou]
2,Simple comme Sylvain [],Hors de prix [Audrey Tautou],Paris-Manhattan []
3,Amour & Turbulences [],La délicatesse [Audrey Tautou],La délicatesse [Audrey Tautou]
4,Lol [],Simple comme Sylvain [],Hors de prix [Audrey Tautou]
5,L'arnacoeur [],Amour & Turbulences [],De vrais mensonges [Audrey Tautou]
6,"Un peu, beaucoup, aveuglément []",L'arnacoeur [],L'ami de mon amie []
7,Fanfan [],Lol [],L'auberge espagnole [Audrey Tautou]
8,Un bonheur n'arrive jamais seul [],"Un peu, beaucoup, aveuglément []",Quatre aventures de Reinette et Mirabelle []
9,L'amour dure trois ans [],Le battement d'ailes du papillon [Audrey Tautou],L'arnacoeur []


Mission Cléopâtre — Gérard Depardieu, Christian Clavier, Jamel Debbouze, Monica Bellucci


,"poids 0,4","poids 0,7",poids 1
0,"Astérix & Obélix contre César [Christian Clavier, Gérard Depardieu]","Astérix & Obélix contre César [Christian Clavier, Gérard Depardieu]","Astérix & Obélix contre César [Christian Clavier, Gérard Depardieu]"
1,Le Petit Nicolas [],Le Petit Nicolas [],Le Petit Nicolas []
2,Les vacances du petit Nicolas [],La vache [Jamel Debbouze],La vache [Jamel Debbouze]
3,La nouvelle guerre des boutons [],Les vacances du petit Nicolas [],Astérix: Le Secret de la potion magique [Christian Clavier]
4,Un cadeau du ciel [],Astérix: Le Secret de la potion magique [Christian Clavier],Les triplettes de Belleville []
5,Astérix et la Surprise de César [],Les triplettes de Belleville [],Le grand méchant renard []
6,Ernest et Célestine [],Un cadeau du ciel [],1981 []
7,Astérix et les Vikings [],Le grand méchant renard [],La Chèvre [Gérard Depardieu]
8,La Guerre des boutons [],La nouvelle guerre des boutons [],Princes et Princesses []
9,Belle et Sébastien [],Astérix et la Surprise de César [],La Guerre des tuques []


> **Constat** : à 0,4, le bloc acteurs reste discret. À 0,7, il fait entrer des films choisis pour leurs acteurs, souvent pertinents : Hors de prix, La délicatesse et Le battement d'ailes du papillon (Audrey Tautou) pour Amélie, V pour Vendetta et Captain America: First Avenger (Hugo Weaving) pour Matrix, Samba (Omar Sy) pour Intouchables, La vache (Jamel Debbouze) pour Mission Cléopâtre. Mais des films sans aucun acteur commun entrent aussi : Cube et Les Clowns tueurs venus d'ailleurs pour Alien, Space Jam et Les Simpson pour Toy Story. À 1, ces intrus se multiplient (Elmer le remue-méninges, Psycho Goreman, Bad Taste pour Alien ; Snoopy, Abominable pour Toy Story) et la liste d'Amélie devient une filmographie d'Audrey Tautou. Les sagas (Taxi, La Communauté de l'anneau, Matrix) gardent leurs suites en tête à tous les poids.

> **Décision** : retenir un poids de 0,7 pour le bloc acteurs. Il apporte des recommandations « par acteur » souvent pertinentes (Audrey Tautou pour Amélie, Hugo Weaving pour Matrix, Omar Sy pour Intouchables), au prix de quelques films inattendus. Le poids 1 est écarté : les intrus s'y multiplient. Ce réglage sera à vérifier sur la table de mixage de l'appli.

### Bilan de la partie 6

> **Constat** : sur les huit films tests, de la version 3 (moteur actuel) à la version finale (réalisateurs 0,4, acteurs 0,7), sur les 10 plus proches voisins :
>
> | Film | Version 3 | Version finale |
> |---|---|---|
> | Alien | 4 films de la saga | 4 films de la saga, plus Cube et Les Clowns tueurs venus d'ailleurs (sans lien) |
> | Taxi | Taxi 2 en tête | Taxi 2 en tête |
> | Toy Story | 3 suites et des films Pixar | 3 suites en tête, Cars et 1001 Pattes (John Lasseter), Space Jam et Les Simpson (sans lien) |
> | La Communauté de l'anneau | 2 suites, 3 Hobbit dispersés | 2 suites, puis les 3 Hobbit dans les six premiers |
> | Matrix | 3 suites, puis des super-héros | 3 suites, V pour Vendetta et Captain America: First Avenger (Hugo Weaving) |
> | Intouchables | des films francophones | Hors normes et Samba (Toledano et Nakache), Les Petits Mouchoirs (François Cluzet) |
> | Amélie | des comédies françaises | trois films d'Audrey Tautou (Hors de prix, La délicatesse, Le battement d'ailes du papillon) |
> | Mission Cléopâtre | 3 Astérix | 3 Astérix, Un cadeau du ciel (Alain Chabat), La vache (Jamel Debbouze) |
>
> Les réalisateurs et les acteurs ajoutent ce que ni les mots-clés ni les genres ne voyaient : l'auteur et les interprètes d'un film. Les sagas restent en tête. La contrepartie : quelques films inattendus, liés aux films « orphelins » (réalisateur ou acteurs absents du reste du catalogue).
>
> **Décision** : le moteur final ajoute deux blocs TF-IDF à la version 3 : réalisateurs (poids 0,4) et acteurs (4 têtes d'affiche, poids 0,7). Prochaine étape : les reporter dans `reco.py`.

## 7. Mise à jour du fichier préparé

> **Objectif** : ajouter au fichier `data/films_prepares.csv` les codes des réalisateurs (`directors`) et des quatre têtes d'affiche (`acteurs`), dont `reco.py` aura besoin, ainsi que leurs noms (`realisation`, `casting`) pour l'affichage dans l'appli. Le fichier est réécrit en entier.

In [213]:
films["directors"] = realisateurs["directors"]
films["acteurs"] = acteurs_films["nconst"]
films.to_csv("../data/films_prepares.csv", index=False)

verif = pd.read_csv("../data/films_prepares.csv")
print(verif.shape)
verif[["title", "directors", "acteurs", "realisation", "casting"]].head(3)

(6942, 19)


,title,directors,acteurs,realisation,casting
0,Kate et Léopold,nm0003506,"nm0000212,nm0413168,nm0000630,nm0005227",James Mangold,"Meg Ryan, Hugh Jackman"
1,Le boucher,nm0001031,"nm0000804,nm0946179,nm0946179,nm0664207",Claude Chabrol,"Stéphane Audran, Jean Yanne"
2,Le cerveau d'acier,nm0765121,"nm0000967,nm0164540,nm0684521,nm0769974",Joseph Sargent,"Eric Braeden, Susan Clark"


> **Constat** : le fichier préparé compte maintenant 6 942 films et 19 colonnes : les 15 d'avant, plus `directors`, `acteurs`, `realisation` et `casting`.

## 8. Du notebook à l'appli

> **Objectif** : réunir le moteur retenu (version 3, durée à 0, plus les réalisateurs au poids 0,4 et les acteurs au poids 0,7) dans un fichier `reco.py`, sous forme de fonctions que l'appli Streamlit pourra importer. Le notebook a servi à explorer et à décider ; `reco.py` ne garde que le nécessaire.

> **Objectif** : vérifier que `reco.py` reproduit exactement le moteur retenu. On recharge le catalogue depuis le fichier CSV, comme le fera l'appli, puis on prépare les caractéristiques et le modèle avec les fonctions de `reco.py`.

In [214]:
import sys
sys.path.append("../scripts")
import reco

films_reco = pd.read_csv("../data/films_prepares.csv")
X_reco = reco.preparer_caracteristiques(films_reco)
modele_reco = reco.creer_modele(X_reco)
X_reco.shape

(6942, 12236)

> **Objectif** : comparer, pour les huit films tests, les 10 recommandations de `reco.py` et celles de la version finale du notebook (`X5_07` : réalisateurs 0,4, acteurs 0,7).

In [215]:
for titre in films_tests:
    t = films_tests[titre]
    titres_reco = list(reco.recommander(t, films_reco, X_reco, modele_reco, nb=10)["title"])
    i = films[films["tconst"] == t].index[0]
    distances, indices = modele5_07.kneighbors(X5_07.iloc[[i]], n_neighbors=11)
    titres_notebook = list(films.iloc[indices[0][1:]]["title"])
    print(titre, ":", titres_reco == titres_notebook)

Alien : True
Taxi : True
Toy Story : True
La Communauté de l'anneau : True
Matrix : True
Intouchables : True
Amélie : True
Mission Cléopâtre : True


> **Constat** : pour les huit films tests, `reco.py` donne exactement les mêmes 10 recommandations que la version finale du notebook (réalisateurs 0,4, acteurs 0,7). Le moteur complet tient dans trois fonctions (`preparer_caracteristiques`, `creer_modele`, `recommander`), prêtes à être importées par l'appli.

> **Remarque** : le notebook garde en mémoire la version de `reco.py` chargée par `import reco`. Après une modification de `reco.py`, il faut redémarrer le noyau, puis réexécuter les cellules.

In [216]:
from datetime import datetime

print("Exécuté le", datetime.now().strftime("%d/%m/%Y à %H:%M:%S"))

Exécuté le 25/11/2026 à 20:31:09
